In [ ]:
# 0

# --- Standard library ---
import warnings
from pathlib import Path
from itertools import combinations

# --- Data / numeric ---
import pandas as pd
import numpy as np

# --- Statistics ---
from scipy import stats
import statsmodels.formula.api as smf

# --- ML / regression (scikit-learn) ---
from sklearn.linear_model import lasso_path
from sklearn.exceptions import ConvergenceWarning

# --- Parallel ---
from joblib import Parallel, delayed

# --- Warning filters ---
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [2]:
# 1

XLSX = 'kospi_data.xlsx'
FIN_OUT = 'financials.parquet'
DEL_OUT = 'delisted.parquet'

EXCLUDE_FIN = ['A000680', 'A000880', 'A005110', 'A008060', 'A015020', 'A023590',
               'A026890', 'A030190', 'A034310', 'A210980', 'A244920']

SHEET_ITEM = {
    '01_actq': 'actq', '02_rectq': 'rectq', '03_invtq': 'invtq',
    '04_ppentq': 'ppentq', '05_atq': 'atq', '06_lctq': 'lctq',
    '07_dlttq': 'dlttq', '08_dlcq': 'dlcq', '09_ltq': 'ltq',
    '10_seqq': 'seqq', '11_cogsq': 'cogsq', '12_xsgaq': 'xsgaq',
    '13_saleq': 'saleq', '14_apq': 'apq', '15_opinc': 'opinc',
    '16_xintq': 'xintq', '17_ibq': 'ibq', '18_txditcq': 'txditcq',
    '19_pstkq': 'pstkq', '20_cheq': 'cheq', '21_oancfq': 'oancfq',
    '22_dpq': 'dpq', '23_dvq': 'dvq', '24_cstkq': 'cstkq',
    '25_req': 'req', '26_epsq': 'epsq',
}

def load_sheet(path, sheet, var):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    periods = raw.iloc[9, 3:].tolist()
    body = raw.iloc[13:, :].reset_index(drop=True)
    codes = body.iloc[:, 0].astype(str)
    mask = codes.str.startswith('A')
    codes = codes[mask].reset_index(drop=True)
    vals = body.loc[mask.values, 3:].apply(pd.to_numeric, errors='coerce').reset_index(drop=True)
    vals.columns = periods
    vals.insert(0, 'code', codes)
    long = vals.melt(id_vars='code', var_name='period', value_name=var)
    long['date'] = pd.to_datetime(long['period'].astype(int).astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
    return long[['code', 'date', var]]

def build_financials(path=XLSX, out=FIN_OUT):
    if Path(out).exists():
        fin = pd.read_parquet(out)
        fin = fin[~fin['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)
        print(f'{out} loaded. shape {fin.shape}  n_comp {fin["code"].nunique()}')
        return fin
    fin = None
    for sheet, var in SHEET_ITEM.items():
        long = load_sheet(path, sheet, var)
        fin = long if fin is None else fin.merge(long, on=['code', 'date'], how='outer')
    fin = fin[~fin['code'].isin(EXCLUDE_FIN)]
    fin = fin.sort_values(['code', 'date']).reset_index(drop=True)
    fin.to_parquet(out)
    print(f'{out} written. shape {fin.shape}  n_comp {fin["code"].nunique()}  n_date {fin["date"].nunique()}')
    return fin

def build_delisted(path=XLSX, out=DEL_OUT):
    if Path(out).exists():
        df = pd.read_parquet(out)
        df = df[~df['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)
        print(f'{out} loaded. {len(df)}개')
        return df
    raw = pd.read_excel(path, sheet_name='34_delisted', header=None)
    body = raw.iloc[13:, :].reset_index(drop=True)
    codes = body.iloc[:, 0].astype(str)
    mask = codes.str.startswith('A')
    df = pd.DataFrame({
        'code': codes[mask].values,
        'name': body.loc[mask.values, 1].astype(str).values,
        'del_date': pd.to_datetime(body.loc[mask.values, 3].astype('Int64').astype(str),
                                   format='%Y%m%d', errors='coerce').values,
        'reason': body.loc[mask.values, 4].astype(str).values,
    })
    df = df[~df['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)
    r = df['reason'].fillna('')
    df['reason_cat'] = np.select(
        [r.str.contains('합병|해산'), r.str.contains('자본전액잠식|자본잠식'),
         r.str.contains('감사의견|의견거절'), r.str.contains('회사정리|정리절차|회생'),
         r.str.contains('신청|자진')],
        ['합병·해산', '자본잠식', '감사의견', '정리절차', '자진·신청'], default='기타')
    df.to_parquet(out)
    print(f'{out} written. {len(df)}개')
    return df

financials = build_financials()
delisted = build_delisted()
print('columns:', financials.columns.tolist())

financials.parquet loaded. shape (116235, 28)  n_comp 1107
delisted.parquet loaded. 355개
columns: ['code', 'date', 'actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlttq', 'dlcq', 'ltq', 'seqq', 'cogsq', 'xsgaq', 'saleq', 'apq', 'opinc', 'xintq', 'ibq', 'txditcq', 'pstkq', 'cheq', 'oancfq', 'dpq', 'dvq', 'cstkq', 'req', 'epsq']


In [3]:
# 2

FVOL_COLS = ['actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlcq',
             'apq', 'dlttq', 'ltq', 'seqq', 'xoprq', 'cogsq', 'xsgaq']

def compute_fvol(df, deflator='saleq', fvol_cols=FVOL_COLS, diff_lag=4, win=8, min_obs=4):
    d = df.sort_values(['code', 'date']).reset_index(drop=True).copy()
    d['xoprq'] = d['cogsq'] + d['xsgaq']

    pq = pd.PeriodIndex(d['date'], freq='Q')
    d['qidx'] = pq.year * 4 + (pq.quarter - 1)

    g = d.groupby('code', sort=False)
    has_lag = (d['qidx'] - g['qidx'].shift(diff_lag)) == diff_lag
    for c in fvol_cols:
        d[f'd_{c}'] = np.where(has_lag, d[c] - g[c].shift(diff_lag), np.nan)

    g = d.groupby('code', sort=False)
    for c in fvol_cols:
        d[f'std_{c}'] = (g[f'd_{c}'].apply(lambda s: s.rolling(win, min_periods=min_obs).std())
                         .reset_index(level=0, drop=True))

    d['defl_pos'] = d[deflator].where(d[deflator] > 0)
    g = d.groupby('code', sort=False)
    d['avg_defl'] = (g['defl_pos'].apply(lambda s: s.rolling(win, min_periods=min_obs).mean())
                     .reset_index(level=0, drop=True))
    d['avg_defl'] = d['avg_defl'].where(d['avg_defl'] > 0)

    ind = []
    for c in fvol_cols:
        d[f'fvol_{c}'] = d[f'std_{c}'] / d['avg_defl']
        ind.append(f'fvol_{c}')
    return d, ind

def finalize_fvol(d, ind, fvol_cols=FVOL_COLS, min_valid=10, suffix=''):
    d = d.copy()
    rcols = [f'rank_{c}' for c in fvol_cols]
    ranked = d.groupby('date')[ind].rank(pct=True)
    ranked.columns = rcols
    d = pd.concat([d, ranked], axis=1)
    d['n_valid'] = d[rcols].notna().sum(axis=1)
    col = f'FVOL{suffix}'
    d[col] = d[rcols].mean(axis=1)
    d.loc[d['n_valid'] < min_valid, col] = np.nan
    return d, col

financials = pd.read_parquet('financials.parquet')

fv_sale, ind = compute_fvol(financials, deflator='saleq')
fv_sale, col_s = finalize_fvol(fv_sale, ind, suffix='_sale')

fv_asset, ind = compute_fvol(financials, deflator='atq')
fv_asset, col_a = finalize_fvol(fv_asset, ind, suffix='_asset')

fvol = fv_sale[['code', 'date', col_s]].merge(
    fv_asset[['code', 'date', col_a]], on=['code', 'date'])

for c in [col_s, col_a]:
    v = fvol[fvol[c].notna()]
    print(f'{c}: valid {len(v)}  avg firms/q {v.groupby("date").size().mean():.0f}  '
          f'range {v["date"].min().date()}~{v["date"].max().date()}')
    print(f'  describe: {v[c].describe()[["mean","std","min","50%","max"]].round(3).to_dict()}')

corr = fvol[[col_s, col_a]].corr().iloc[0, 1]
print(f'매출분모 FVOL vs 자산분모 FVOL 상관: {corr:.3f}')
print('분기당 종목수 (매출분모):')
gq = fvol[fvol[col_s].notna()].groupby('date').size()
display(gq.iloc[[0, len(gq)//4, len(gq)//2, 3*len(gq)//4, -1]]
        .rename_axis('date').reset_index(name='종목수'))

FVOL_sale: valid 66765  avg firms/q 681  range 2001-12-31~2026-03-31
  describe: {'mean': 0.501, 'std': 0.203, 'min': 0.021, '50%': 0.483, 'max': 1.0}
FVOL_asset: valid 66828  avg firms/q 682  range 2001-12-31~2026-03-31
  describe: {'mean': 0.501, 'std': 0.189, 'min': 0.022, '50%': 0.49, 'max': 1.0}
매출분모 FVOL vs 자산분모 FVOL 상관: 0.713
분기당 종목수 (매출분모):


,date,종목수
0,2001-12-31,557
1,2007-12-31,639
2,2014-03-31,675
3,2020-03-31,729
4,2026-03-31,763


In [4]:
# 3

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

rcols = [f'rank_{c}' for c in FVOL_COLS]
valid = fv[fv['FVOL'].notna()].copy()

panelA = []
for c in FVOL_COLS:
    s = fv[f'fvol_{c}'].replace([np.inf, -np.inf], np.nan).dropna()
    panelA.append({'item': c, 'mean': s.mean(), 'std': s.std(),
                   'median': s.median(), 'N': len(s)})
print('Panel A: 14개 개별 측정치 분포 (fvol_k 원값)')
display(pd.DataFrame(panelA).round(3))

panelB = []
for c in FVOL_COLS:
    corr = valid[[f'rank_{c}', 'FVOL']].corr().iloc[0, 1]
    panelB.append({'item': c, 'corr_with_FVOL': corr})
print('Panel B: 각 측정치 rank와 FVOL의 상관 (공통성분)')
display(pd.DataFrame(panelB).round(3))

rank_corr = valid[rcols].corr()
mask = ~np.eye(14, dtype=bool)
print(f'14개 측정치 rank 간 평균 Spearman 상관: 평균 {rank_corr.values[mask].mean():.3f}  '
      f'최소 {rank_corr.values[mask].min():.3f}  최대 {rank_corr.values[mask].max():.3f}')

short = [c.replace('q', '') for c in FVOL_COLS]
rc = rank_corr.copy()
rc.index = short
rc.columns = short
print('상관행렬 (rank 간)')
display(rc.round(2))

Panel A: 14개 개별 측정치 분포 (fvol_k 원값)


,item,mean,std,median,N
0,actq,0.566,1.920,0.248,66792
1,rectq,0.226,0.716,0.114,66780
2,invtq,0.137,0.315,0.079,64601
3,ppentq,0.338,3.206,0.098,66764
4,atq,0.834,3.972,0.313,66792
5,lctq,0.640,6.468,0.259,66792
6,dlcq,0.340,1.339,0.162,66792
7,apq,0.259,5.893,0.110,66760
8,dlttq,0.220,0.788,0.089,44357
9,ltq,0.825,19.774,0.255,66792


Panel B: 각 측정치 rank와 FVOL의 상관 (공통성분)


,item,corr_with_FVOL
0,actq,0.823
1,rectq,0.708
2,invtq,0.576
3,ppentq,0.561
4,atq,0.852
5,lctq,0.820
6,dlcq,0.632
7,apq,0.706
8,dlttq,0.485
9,ltq,0.857


14개 측정치 rank 간 평균 Spearman 상관: 평균 0.440  최소 0.126  최대 0.914
상관행렬 (rank 간)


,act,rect,invt,ppent,at,lct,dlc,ap,dltt,lt,se,xopr,cogs,xsga
act,1.00,0.61,0.45,0.38,0.79,0.67,0.42,0.57,0.25,0.73,0.60,0.53,0.47,0.47
rect,0.61,1.00,0.40,0.23,0.53,0.50,0.31,0.58,0.20,0.52,0.41,0.60,0.56,0.38
invt,0.45,0.40,1.00,0.24,0.39,0.37,0.35,0.32,0.17,0.39,0.31,0.45,0.43,0.27
ppent,0.38,0.23,0.24,1.00,0.52,0.39,0.36,0.28,0.35,0.47,0.46,0.26,0.23,0.29
at,0.79,0.53,0.39,0.52,1.00,0.70,0.48,0.54,0.33,0.85,0.72,0.50,0.44,0.46
lct,0.67,0.50,0.37,0.39,0.70,1.00,0.71,0.56,0.44,0.80,0.53,0.46,0.41,0.44
dlc,0.42,0.31,0.35,0.36,0.48,0.71,1.00,0.29,0.51,0.58,0.37,0.29,0.27,0.30
ap,0.57,0.58,0.32,0.28,0.54,0.56,0.29,1.00,0.21,0.57,0.42,0.56,0.52,0.40
dltt,0.25,0.20,0.17,0.35,0.33,0.44,0.51,0.21,1.00,0.40,0.29,0.14,0.13,0.23
lt,0.73,0.52,0.39,0.47,0.85,0.80,0.58,0.57,0.40,1.00,0.59,0.51,0.45,0.45


In [5]:
# 4

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

d = fv[['code', 'date', 'FVOL']].dropna().copy()
d['qidx'] = pd.PeriodIndex(d['date'], freq='Q').year * 4 + (pd.PeriodIndex(d['date'], freq='Q').quarter - 1)
d = d.sort_values(['code', 'qidx']).reset_index(drop=True)

print('=== FVOL 자기상관 (lag별) ===')
print('같은 종목의 FVOL(t)와 FVOL(t+k) 상관:')
r_ac = []
for lag in [1, 2, 4, 8, 12, 20]:
    g = d.copy()
    g['fut_qidx'] = g['qidx'] + lag
    merged = g.merge(g[['code','qidx','FVOL']].rename(columns={'qidx':'fut_qidx','FVOL':'FVOL_fut'}),
                     on=['code','fut_qidx'], how='inner')
    corr = merged[['FVOL','FVOL_fut']].corr().iloc[0,1]
    r_ac.append({'lag(분기)': lag, '년': lag/4, '상관': corr, 'n': len(merged)})
display(pd.DataFrame(r_ac).round({'년': 1, '상관': 3}))

print('=== 분위 유지율 (transition) ===')
print('이번 분기 decile이 k분기 후에도 같은/인접 decile에 남는 비율:')
d['dec'] = d.groupby('date')['FVOL'].transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop'))
r_tr = []
for lag in [1, 4, 8]:
    g = d.copy()
    g['fut_qidx'] = g['qidx'] + lag
    m = g.merge(g[['code','qidx','dec']].rename(columns={'qidx':'fut_qidx','dec':'dec_fut'}),
                on=['code','fut_qidx'], how='inner')
    same = (m['dec'] == m['dec_fut']).mean()
    adj = (abs(m['dec'] - m['dec_fut']) <= 1).mean()
    r_tr.append({'lag(분기)': lag, '동일decile%': same*100, '±1이내%': adj*100})
display(pd.DataFrame(r_tr).round(1))

print('=== 극단 분위의 지속성 (전이 확률) ===')
print('최저(0)·최고(9) decile이 4분기 후 어디로 가는지:')
g = d.copy(); g['fut_qidx'] = g['qidx'] + 4
m = g.merge(g[['code','qidx','dec']].rename(columns={'qidx':'fut_qidx','dec':'dec_fut'}),
            on=['code','fut_qidx'], how='inner')
r_ext = []
for start in [0, 9]:
    sub = m[m['dec'] == start]['dec_fut']
    r_ext.append({'시작decile': start, '4분기후_평균decile': sub.mean(), '같은곳유지%': (sub==start).mean()*100})
display(pd.DataFrame(r_ext).round({'4분기후_평균decile': 2, '같은곳유지%': 1}))

=== FVOL 자기상관 (lag별) ===
같은 종목의 FVOL(t)와 FVOL(t+k) 상관:


,lag(분기),년,상관,n
0,1,0.2,0.973,65802
1,2,0.5,0.933,64842
2,4,1.0,0.843,62929
3,8,2.0,0.667,59209
4,12,3.0,0.547,55584
5,20,5.0,0.457,48687


=== 분위 유지율 (transition) ===
이번 분기 decile이 k분기 후에도 같은/인접 decile에 남는 비율:


,lag(분기),동일decile%,±1이내%
0,1,64.4,95.5
1,4,35.0,71.7
2,8,23.9,55.4


=== 극단 분위의 지속성 (전이 확률) ===
최저(0)·최고(9) decile이 4분기 후 어디로 가는지:


,시작decile,4분기후_평균decile,같은곳유지%
0,0,0.81,59.1
1,9,8.49,70.1


In [6]:
# 5

ALL_ITEMS = ['actq','rectq','invtq','ppentq','atq','lctq','dlttq','dlcq','ltq',
             'seqq','cogsq','xsgaq','saleq','apq','opinc','xintq','ibq','txditcq',
             'pstkq','cheq','oancfq','dpq','dvq','cstkq','req','epsq']

print('=== 1. 항목별 결측률 (FVOL 14개 표시) ===')
FVOL14 = ['actq','rectq','invtq','ppentq','atq','lctq','dlcq','apq','dlttq','ltq','seqq','cogsq','xsgaq']
r_miss = []
for c in ALL_ITEMS:
    nan = financials[c].isna().mean()
    r_miss.append({'항목': c, '결측률%': nan*100, 'FVOL': c in FVOL14})
display(pd.DataFrame(r_miss).round({'결측률%': 1}))

print('=== 2. 폐지 종목 실태 (공식 상장폐지 목록 기준) ===')
fin_has = financials.groupby('code')['atq'].apply(lambda s: s.notna().any())
fin_codes = set(fin_has[fin_has].index)
del_all = delisted.copy()
has_fin = del_all['code'].isin(fin_codes)
print(f'공식 상장폐지 종목: {len(del_all)}개')
print(f'  재무 데이터 존재: {has_fin.sum()}개 (FVOL 분석 대상)')
print(f'  재무 데이터 없음: {(~has_fin).sum()}개 (대부분 1999년 이전 폐지)')

dfin = del_all[has_fin].copy()
print(f'폐지 시점 분포 (재무 존재 {len(dfin)}개, 공식 폐지일 기준, 연도별):')
display(dfin['del_date'].dt.year.value_counts().sort_index()
        .rename_axis('연도').reset_index(name='폐지수'))

print(f'폐지 사유 분포 (재무 존재 {len(dfin)}개):')
display(dfin['reason_cat'].value_counts()
        .rename_axis('사유').reset_index(name='종목수'))

print('=== 3. 종목별 데이터 이력 길이 ===')
life = financials[financials['atq'].notna()].groupby('code')['date'].agg(['min','max','count'])
print('데이터 존재 분기 수 분포:')
display(life['count'].describe()[['min','25%','50%','75%','max']].round(0)
        .rename_axis('통계').reset_index(name='분기수'))
print(f'  8분기 미만(측정 워밍업 부족) 종목: {(life["count"]<8).sum()}')

print('=== 4. 차분 이상치 점검 (A6: 구조적 단절) ===')
d = financials.sort_values(['code','date']).copy()
d['atq_chg'] = d.groupby('code')['atq'].pct_change(4)
extreme = d['atq_chg'].abs() > 2
print(f'자산총계 4분기 변화율 |>200%| 관측치: {extreme.sum()} ({extreme.mean():.2%})')
print(f'  변화율 분포: {d["atq_chg"].describe()[["min","25%","50%","75%","max"]].round(2).to_dict()}')

=== 1. 항목별 결측률 (FVOL 14개 표시) ===


,항목,결측률%,FVOL
0,actq,36.2,True
1,rectq,36.2,True
2,invtq,38.2,True
3,ppentq,36.2,True
4,atq,36.2,True
5,lctq,36.2,True
6,dlttq,54.9,True
7,dlcq,36.2,True
8,ltq,36.2,True
9,seqq,36.2,True


=== 2. 폐지 종목 실태 (공식 상장폐지 목록 기준) ===
공식 상장폐지 종목: 355개
  재무 데이터 존재: 235개 (FVOL 분석 대상)
  재무 데이터 없음: 120개 (대부분 1999년 이전 폐지)
폐지 시점 분포 (재무 존재 235개, 공식 폐지일 기준, 연도별):


,연도,폐지수
0,2000,3
1,2001,16
2,2002,22
3,2003,12
4,2004,15
5,2005,15
6,2006,4
7,2007,9
8,2008,2
9,2009,17


폐지 사유 분포 (재무 존재 235개):


,사유,종목수
0,감사의견,70
1,합병·해산,67
2,기타,53
3,자본잠식,21
4,자진·신청,15
5,정리절차,9


=== 3. 종목별 데이터 이력 길이 ===
데이터 존재 분기 수 분포:


,통계,분기수
0,min,1.0
1,25%,44.0
2,50%,94.0
3,75%,105.0
4,max,105.0


  8분기 미만(측정 워밍업 부족) 종목: 31
=== 4. 차분 이상치 점검 (A6: 구조적 단절) ===
자산총계 4분기 변화율 |>200%| 관측치: 371 (0.32%)
  변화율 분포: {'min': -0.98, '25%': -0.02, '50%': 0.05, '75%': 0.13, 'max': 25.81}


In [7]:
# 6

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

d = fv.sort_values(['code','date']).copy()

g = d.groupby('code', sort=False)
d['atq_lag1'] = g['atq'].shift(1)
d['qbe'] = d['seqq'].fillna(0) + d['txditcq'].fillna(0) - d['pstkq'].fillna(0)
d['qbe_lag1'] = g['qbe'].shift(1)

d['GP'] = (d['saleq'] - d['cogsq']) / d['atq_lag1']
d['OP'] = (d['saleq'] - d['cogsq'] - d['xsgaq'] - d['xintq']) / d['qbe_lag1']
d['ROE'] = d['ibq'] / d['qbe_lag1']

for c in ['GP','OP','ROE']:
    d[c] = d[c].replace([np.inf,-np.inf], np.nan)

v = d[d['FVOL'].notna()].copy()
v['dec'] = v.groupby('date')['FVOL'].transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop')) + 1

print('=== FVOL decile별 수익성 (분기별 중앙값의 평균) ===')
print('decile 1=최저FVOL(롱), 10=최고FVOL(숏)')
tbl = {}
for c in ['GP','OP','ROE']:
    med = v.groupby(['date','dec'])[c].median().reset_index()
    tbl[c] = med.groupby('dec')[c].mean()
cnt = v.groupby('dec').size()
r_dec = []
for dc in range(1, 11):
    r_dec.append({'decile': dc, 'GP': tbl['GP'][dc], 'OP': tbl['OP'][dc],
                  'ROE': tbl['ROE'][dc], 'n': int(cnt[dc])})
display(pd.DataFrame(r_dec).round(3))

print('=== 단조성 검정 (Low decile 1 vs High decile 10) ===')
r_mono = []
for c in ['GP','OP','ROE']:
    lo, hi = tbl[c][1], tbl[c][10]
    monotone = all(tbl[c][i] >= tbl[c][i+1] for i in range(1, 10))
    n_desc = sum(tbl[c][i] >= tbl[c][i+1] for i in range(1, 10))
    r_mono.append({'지표': c, '저FVOL': lo, '고FVOL': hi, '차이': lo - hi,
                   '완전단조': monotone, '감소구간': f'{n_desc}/9'})
display(pd.DataFrame(r_mono).round(3))

print('=== 자산분모 FVOL로도 동일 확인 ===')
fv2, ind2 = compute_fvol(financials, deflator='atq')
fv2, _ = finalize_fvol(fv2, ind2, suffix='')
d2 = d.merge(fv2[['code','date','FVOL']].rename(columns={'FVOL':'FVOL_a'}), on=['code','date'])
v2 = d2[d2['FVOL_a'].notna()].copy()
v2['dec'] = v2.groupby('date')['FVOL_a'].transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop')) + 1
r_asset = []
for c in ['GP','OP','ROE']:
    m = v2.groupby(['date','dec'])[c].median().reset_index().groupby('dec')[c].mean()
    r_asset.append({'지표': c, '저(자산분모)': m[1], '고(자산분모)': m[10], '차이': m[1]-m[10]})
display(pd.DataFrame(r_asset).round(3))

=== FVOL decile별 수익성 (분기별 중앙값의 평균) ===
decile 1=최저FVOL(롱), 10=최고FVOL(숏)


,decile,GP,OP,ROE,n
0,1,0.054,0.023,0.022,6716
1,2,0.046,0.020,0.019,6673
2,3,0.043,0.018,0.018,6665
3,4,0.039,0.016,0.016,6669
4,5,0.037,0.014,0.014,6682
5,6,0.035,0.012,0.013,6653
6,7,0.033,0.010,0.011,6661
7,8,0.030,0.006,0.008,6673
8,9,0.026,0.001,0.004,6665
9,10,0.017,-0.015,-0.015,6708


=== 단조성 검정 (Low decile 1 vs High decile 10) ===


,지표,저FVOL,고FVOL,차이,완전단조,감소구간
0,GP,0.054,0.017,0.037,True,9/9
1,OP,0.023,-0.015,0.038,True,9/9
2,ROE,0.022,-0.015,0.037,True,9/9


=== 자산분모 FVOL로도 동일 확인 ===


,지표,저(자산분모),고(자산분모),차이
0,GP,0.034,0.029,0.005
1,OP,0.014,-0.001,0.015
2,ROE,0.016,-0.001,0.017


In [8]:
# 7

PRICE_OUT = 'prices.parquet'
MARKET_OUT = 'market.parquet'

PRICE_SHEETS = {
    '27_return': 'ret',
    '28_mktcap': 'mktcap',
    '29_price': 'price',
    '30_dret': 'dret',
}

MARKET_SHEETS = {
    '31_kret': 'kret',
    '32_rf': 'rf',
}

def load_wide_sheet(path, sheet, var):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    codes = raw.iloc[7, 1:].astype(str)
    mask = codes.str.startswith('A')
    codes = codes[mask]
    dates = pd.to_datetime(raw.iloc[14:, 0].tolist())
    vals = raw.iloc[14:, 1:].loc[:, mask.values].apply(pd.to_numeric, errors='coerce')
    vals.columns = codes.values
    vals.index = dates
    long = vals.stack().rename(var).reset_index()
    long.columns = ['date', 'code', var]
    return long[['code', 'date', var]]

def load_single_sheet(path, sheet, var):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    dates = pd.to_datetime(raw.iloc[14:, 0].tolist())
    vals = pd.to_numeric(raw.iloc[14:, 1], errors='coerce')
    return pd.DataFrame({'date': dates, var: vals.values})

def build_prices(path=XLSX, out=PRICE_OUT):
    if Path(out).exists():
        df = pd.read_parquet(out)
        df = df[~df['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)
        print(f'{out} loaded. shape {df.shape}  n_comp {df["code"].nunique()}')
        return df
    monthly = None
    for sheet, var in [('27_return', 'ret'), ('28_mktcap', 'mktcap'), ('29_price', 'price')]:
        long = load_wide_sheet(path, sheet, var)
        monthly = long if monthly is None else monthly.merge(long, on=['code', 'date'], how='outer')
    monthly = monthly[~monthly['code'].isin(EXCLUDE_FIN)]
    monthly = monthly.sort_values(['code', 'date']).reset_index(drop=True)
    monthly.to_parquet(out)
    print(f'{out} written. shape {monthly.shape}  n_comp {monthly["code"].nunique()}  n_date {monthly["date"].nunique()}')
    return monthly

def build_daily(path=XLSX, out='daily_returns.parquet'):
    if Path(out).exists():
        df = pd.read_parquet(out)
        df = df[~df['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)
        print(f'{out} loaded. shape {df.shape}  n_comp {df["code"].nunique()}')
        return df
    daily = load_wide_sheet(path, '30_dret', 'dret')
    daily = daily[~daily['code'].isin(EXCLUDE_FIN)]
    daily = daily.dropna(subset=['dret']).sort_values(['code', 'date']).reset_index(drop=True)
    daily.to_parquet(out)
    print(f'{out} written. shape {daily.shape}  n_comp {daily["code"].nunique()}  n_date {daily["date"].nunique()}')
    return daily

def build_market(path=XLSX, out=MARKET_OUT):
    if Path(out).exists():
        df = pd.read_parquet(out)
        print(f'{out} loaded. shape {df.shape}')
        return df
    kret = load_single_sheet(path, '31_kret', 'kret')
    rf = load_single_sheet(path, '32_rf', 'rf')
    market = kret.merge(rf, on='date', how='outer').sort_values('date').reset_index(drop=True)
    market.to_parquet(out)
    print(f'{out} written. shape {market.shape}  range {market["date"].min().date()}~{market["date"].max().date()}')
    return market

prices = build_prices()
daily_returns = build_daily()
market = build_market()

prices.parquet loaded. shape (353133, 5)  n_comp 1107
daily_returns.parquet loaded. shape (4323086, 3)  n_comp 953
market.parquet loaded. shape (6530, 3)


In [9]:
# 8

prices = pd.read_parquet('prices.parquet')

fv_s, ind = compute_fvol(financials, deflator='saleq')
fv_s, col_s = finalize_fvol(fv_s, ind, suffix='_sale')
fv_a, ind = compute_fvol(financials, deflator='atq')
fv_a, col_a = finalize_fvol(fv_a, ind, suffix='_asset')

sig = fv_s[['code', 'date', col_s]].merge(fv_a[['code', 'date', col_a]], on=['code', 'date'])
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')

form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

for col in [col_s, col_a]:
    f = sig[['code', 'form_qtr', col]].dropna().merge(
        form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
    f = f[f['price'] >= 1000].copy()
    f['decile'] = np.ceil(f.groupby('form_qtr')[col].rank(pct=True) * 10).clip(1, 10).astype(int)
    f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

    holds = []
    for k in [1, 2, 3]:
        h = f[['code', 'form_ym', 'decile']].copy()
        h['hold_ym'] = h['form_ym'] + k
        holds.append(h)
    hold = pd.concat(holds, ignore_index=True)

    m = px[['code', 'ym', 'ret']].dropna().merge(
        hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
    m = m.drop_duplicates(['code', 'form_ym', 'ym'])

    ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='ewret')
    wide = ew.pivot(index='ym', columns='decile', values='ewret')
    lh = (wide[1] - wide[10]).dropna()
    t = lh.mean() / (lh.std() / np.sqrt(len(lh)))

    print(f'=== {col} ===')
    print(f'형성 관측치 {len(f)}  분기당 평균 종목 {f.groupby("form_qtr").size().mean():.0f}  '
          f'월수 {len(lh)}  기간 {lh.index.min()}~{lh.index.max()}')
    print(f'EW Low-High: {lh.mean()*100:.2f}%/월  t={t:.2f}')
    print(f'Low(1): {wide[1].mean()*100:.2f}%  High(10): {wide[10].mean()*100:.2f}%')
    print('decile별 평균 월수익률(%):')
    display((wide[list(range(1, 11))].mean() * 100).round(2)
            .rename_axis('decile').reset_index(name='평균월수익%'))

=== FVOL_sale ===
형성 관측치 61778  분기당 평균 종목 630  월수 291  기간 2002-04~2026-06
EW Low-High: 0.79%/월  t=2.80
Low(1): 1.17%  High(10): 0.38%
decile별 평균 월수익률(%):


,decile,평균월수익%
0,1,1.17
1,2,1.35
2,3,1.34
3,4,1.23
4,5,1.17
5,6,1.36
6,7,1.33
7,8,1.03
8,9,0.79
9,10,0.38


=== FVOL_asset ===
형성 관측치 61807  분기당 평균 종목 631  월수 291  기간 2002-04~2026-06
EW Low-High: 0.51%/월  t=1.77
Low(1): 1.17%  High(10): 0.67%
decile별 평균 월수익률(%):


,decile,평균월수익%
0,1,1.17
1,2,1.14
2,3,1.28
3,4,1.35
4,5,1.18
5,6,1.28
6,7,1.20
7,8,0.99
8,9,0.88
9,10,0.67


In [10]:
# 9

prices = pd.read_parquet('prices.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')

form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f = f[f['price'] >= 1000].copy()
f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

holds = []
for k in [1, 2, 3]:
    h = f[['code', 'form_ym', 'decile']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

rp = px[['code', 'ym', 'ret', 'mktcap']].sort_values(['code', 'ym']).copy()
rp['w'] = rp.groupby('code')['mktcap'].shift(1)

m = rp.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
m = m.dropna(subset=['ret', 'w'])
m = m.drop_duplicates(['code', 'form_ym', 'ym'])

vw = (m.groupby(['ym', 'decile'])
      .apply(lambda g: (g['ret'] * g['w']).sum() / g['w'].sum(), include_groups=False)
      .reset_index(name='vwret'))
wide = vw.pivot(index='ym', columns='decile', values='vwret')
lh = (wide[1] - wide[10]).dropna()
t = lh.mean() / (lh.std() / np.sqrt(len(lh)))

print(f'월수 {len(lh)}  기간 {lh.index.min()}~{lh.index.max()}')
print(f'VW Low-High: {lh.mean()*100:.2f}%/월  t={t:.2f}')
print(f'Low(1): {wide[1].mean()*100:.2f}%  High(10): {wide[10].mean()*100:.2f}%')
print('decile별 평균 월수익률(%):')
display((wide[list(range(1, 11))].mean() * 100).round(2)
        .rename_axis('decile').reset_index(name='평균월수익%'))

월수 291  기간 2002-04~2026-06
VW Low-High: 0.17%/월  t=0.38
Low(1): 0.99%  High(10): 0.81%
decile별 평균 월수익률(%):


,decile,평균월수익%
0,1,0.99
1,2,1.59
2,3,1.28
3,4,0.95
4,5,0.71
5,6,1.14
6,7,1.13
7,8,0.40
8,9,0.90
9,10,0.81


In [11]:
# 10

prices = pd.read_parquet('prices.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')

form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f = f[f['price'] >= 1000].copy()
f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

holds = []
for k in range(1, 25):
    h = f[['code', 'form_ym', 'decile']].copy()
    h['hold_ym'] = h['form_ym'] + k
    h['qtr'] = (k - 1) // 3 + 1
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

m = px[['code', 'ym', 'ret']].dropna().merge(
    hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
m = m.drop_duplicates(['code', 'form_ym', 'ym', 'qtr'])

ew = m.groupby(['qtr', 'ym', 'decile'])['ret'].mean().reset_index(name='r')

r_q = []
parts = []
for qtr in range(1, 9):
    w = ew[ew['qtr'] == qtr].pivot(index='ym', columns='decile', values='r')
    lh = (w[1] - w[10]).dropna()
    parts.append(lh.rename(f'q{qtr}'))
    t = lh.mean() / (lh.std() / np.sqrt(len(lh)))
    r_q.append({'Q': f'Q{qtr}', 'L-H%': lh.mean()*100, 't': t,
                'Low%': w[1].mean()*100, 'High%': w[10].mean()*100, '월수': len(lh)})

jt = pd.concat(parts, axis=1)
lh_jt = jt.mean(axis=1).dropna()
t = lh_jt.mean() / (lh_jt.std() / np.sqrt(len(lh_jt)))
r_q.append({'Q': 'JT8', 'L-H%': lh_jt.mean()*100, 't': t,
            'Low%': np.nan, 'High%': np.nan, '월수': len(lh_jt)})
display(pd.DataFrame(r_q).round(2))

,Q,L-H%,t,Low%,High%,월수
0,Q1,0.79,2.80,1.17,0.38,291
1,Q2,0.72,2.74,1.24,0.52,288
2,Q3,0.67,2.52,1.24,0.57,285
3,Q4,0.73,2.70,1.26,0.53,282
4,Q5,0.56,2.05,1.23,0.67,279
5,Q6,0.68,2.52,1.24,0.57,276
6,Q7,0.55,2.03,1.24,0.69,273
7,Q8,0.40,1.44,1.16,0.76,270
8,JT8,0.71,2.66,NaN,NaN,291


In [12]:
# 11

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')

form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

ret_m = px[['code', 'ym', 'ret']].dropna()

def lowhigh(sig, valcol):
    s = sig.dropna(subset=[valcol]).copy()
    s['form_qtr'] = pd.PeriodIndex(s['date'], freq='Q') + 1
    f = s.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
    f = f[f['price'] >= 1000].copy()
    f['decile'] = np.ceil(f.groupby('form_qtr')[valcol].rank(pct=True) * 10).clip(1, 10).astype(int)
    f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')
    holds = []
    for k in [1, 2, 3]:
        h = f[['code', 'form_ym', 'decile']].copy()
        h['hold_ym'] = h['form_ym'] + k
        holds.append(h)
    hold = pd.concat(holds, ignore_index=True)
    m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
    m = m.drop_duplicates(['code', 'form_ym', 'ym'])
    ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='r')
    w = ew.pivot(index='ym', columns='decile', values='r')
    lh = (w[1] - w[10]).dropna()
    t = lh.mean() / (lh.std() / np.sqrt(len(lh)))
    nfirm = f.groupby('form_qtr').size().mean()
    return lh.mean() * 100, t, w[1].mean() * 100, w[10].mean() * 100, nfirm

r_item = []
for c in FVOL_COLS:
    r, t, lo, hi, n = lowhigh(fv[['code', 'date', f'fvol_{c}']], f'fvol_{c}')
    r_item.append({'item': c, 'L-H%': r, 't': t, 'Low%': lo, 'High%': hi, '종목/분기': n})

r, t, lo, hi, n = lowhigh(fv[['code', 'date', 'FVOL']], 'FVOL')
r_item.append({'item': 'FVOL', 'L-H%': r, 't': t, 'Low%': lo, 'High%': hi, '종목/분기': n})
display(pd.DataFrame(r_item).round({'L-H%': 2, 't': 2, 'Low%': 2, 'High%': 2, '종목/분기': 0}))

,item,L-H%,t,Low%,High%,종목/분기
0,actq,0.85,3.39,1.32,0.47,631.0
1,rectq,0.70,2.64,1.17,0.48,631.0
2,invtq,0.14,0.55,0.99,0.86,610.0
3,ppentq,0.45,2.30,1.20,0.75,630.0
4,atq,0.88,3.67,1.32,0.44,631.0
5,lctq,0.94,3.85,1.24,0.30,631.0
6,dlcq,0.69,2.97,1.05,0.36,631.0
7,apq,0.53,2.04,1.04,0.51,630.0
8,dlttq,0.45,1.74,1.17,0.71,420.0
9,ltq,0.81,3.12,1.19,0.39,631.0


In [13]:
# 12

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

MIN_FRAC = 10 / 14

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

rcols = [f'rank_{c}' for c in FVOL_COLS]
sig = fv[['code', 'date'] + rcols].copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

f_base = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f_base = f_base[f_base['price'] >= 1000].reset_index(drop=True)
f_base['form_ym'] = f_base['form_qtr'].dt.asfreq('M', how='end')
f_base['fid'] = np.arange(len(f_base))

R = f_base[rcols].to_numpy()
qtr_codes, _ = pd.factorize(f_base['form_qtr'])
n_qtr = qtr_codes.max() + 1

holds = []
for k in [1, 2, 3]:
    h = f_base[['fid', 'code', 'form_ym']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

ret_m = px[['code', 'ym', 'ret']].dropna()
panel = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
panel = panel.drop_duplicates(['fid', 'ym'])
fid_p = panel['fid'].to_numpy()
ym_codes, ym_uniq = pd.factorize(panel['ym'])
ym_p = ym_codes
ret_p = panel['ret'].to_numpy()
n_ym = len(ym_uniq)

qtr_sorter = np.argsort(qtr_codes, kind='stable')
qtr_sorted = qtr_codes[qtr_sorter]
qtr_starts = np.searchsorted(qtr_sorted, np.arange(n_qtr))
qtr_ends = np.searchsorted(qtr_sorted, np.arange(n_qtr), side='right')

def eval_combo(idx):
    sub = R[:, idx]
    nv = np.sum(~np.isnan(sub), axis=1)
    with np.errstate(invalid='ignore'):
        score = np.nanmean(sub, axis=1)
    score[nv < int(np.ceil(len(idx) * MIN_FRAC))] = np.nan

    dec = np.full(len(score), np.nan)
    for q in range(n_qtr):
        rows = qtr_sorter[qtr_starts[q]:qtr_ends[q]]
        s = score[rows]
        ok = ~np.isnan(s)
        n = ok.sum()
        if n == 0:
            continue
        r = np.empty(n)
        r[np.argsort(s[ok], kind='stable')] = np.arange(1, n + 1)
        d = np.ceil(r / n * 10)
        np.clip(d, 1, 10, out=d)
        tmp = np.full(len(rows), np.nan)
        tmp[ok] = d
        dec[rows] = tmp

    n_form = (~np.isnan(dec)) .sum() / n_qtr

    dec_p = dec[fid_p]
    ok = ~np.isnan(dec_p)
    key = ym_p[ok] * 10 + (dec_p[ok].astype(int) - 1)
    sums = np.bincount(key, weights=ret_p[ok], minlength=n_ym * 10).reshape(n_ym, 10)
    cnts = np.bincount(key, minlength=n_ym * 10).reshape(n_ym, 10)
    with np.errstate(invalid='ignore'):
        mean = np.where(cnts > 0, sums / cnts, np.nan)
    lh = mean[:, 0] - mean[:, 9]
    valid = ~np.isnan(lh)
    lh = lh[valid]
    if len(lh) < 24:
        return np.nan, np.nan, len(lh), np.nan, np.nan, n_form
    lo = np.nanmean(mean[valid, 0]) * 100
    hi = np.nanmean(mean[valid, 9]) * 100
    return (lh.mean() * 100,
            lh.mean() / (lh.std(ddof=1) / np.sqrt(len(lh))),
            len(lh), lo, hi, n_form)

tasks = []
for k in range(1, 15):
    for combo in combinations(range(14), k):
        tasks.append(combo)
print(f'조합 수: {len(tasks)}')

N_JOBS = -1

inc = np.zeros((len(tasks), 14), dtype=bool)
for i, combo in enumerate(tasks):
    inc[i, list(combo)] = True

results = Parallel(n_jobs=N_JOBS)(delayed(eval_combo)(list(combo)) for combo in tasks)

rows = []
for combo, (lh, t, nmo, lo, hi, nf) in zip(tasks, results):
    rows.append({'k': len(combo),
                 'items': '|'.join(FVOL_COLS[j] for j in combo),
                 'excluded': '|'.join(FVOL_COLS[j] for j in range(14) if j not in combo) or '(none)',
                 'lh': lh, 't': t, 'n_month': nmo,
                 'low': lo, 'high': hi, 'n_form': nf})

res = pd.DataFrame(rows)
res.to_parquet('combo_full_all.parquet')
np.save('combo_inc_mask.npy', inc)

print('=== 정합성: full14 ===')
display(res[res['k'] == 14][['lh', 't', 'low', 'high', 'n_form']].round(2))
print('(기존 #8: LH 0.82 t 2.89 Low 1.18 High 0.36 / 형성 639)')

print('=== k별 분포 ===')
display(res.groupby('k').agg(lh_m=('lh', 'mean'), lh_sd=('lh', 'std'),
                             t_m=('t', 'mean'), t_min=('t', 'min'), t_max=('t', 'max'),
                             nform_m=('n_form', 'mean')).round(2))

print('=== k별 최고 조합 (t 기준) ===')
best = res.loc[res.groupby('k')['t'].idxmax()]
display(best[['k', 'excluded', 'lh', 't', 'low', 'high', 'n_form']].round(2))

print('=== k별 최악 조합 (t 기준) ===')
worst = res.loc[res.groupby('k')['t'].idxmin()]
display(worst[['k', 'items', 'lh', 't', 'low', 'high', 'n_form']].round(2))

print('=== t 상위 15 (전체) ===')
display(res.nlargest(15, 't')[['k', 'excluded', 'lh', 't', 'low', 'high']].round(2))
print('=== t 하위 15 (전체) ===')
display(res.nsmallest(15, 't')[['k', 'items', 'lh', 't', 'low', 'high']].round(2))

print('=== lh 상위 10 (전체) ===')
display(res.nlargest(10, 'lh')[['k', 'excluded', 'lh', 't']].round(2))
print('=== lh 하위 10 (전체) ===')
display(res.nsmallest(10, 'lh')[['k', 'items', 'lh', 't']].round(2))

print('=== 항목별 한계기여 (전체: 포함 평균 t − 제외 평균 t) ===')
tv = res['t'].to_numpy()
ok_t = ~np.isnan(tv)
r_contrib = []
for j, c in enumerate(FVOL_COLS):
    m_in = tv[inc[:, j] & ok_t].mean()
    m_out = tv[~inc[:, j] & ok_t].mean()
    r_contrib.append({'item': c, '포함': m_in, '제외': m_out, '차이': m_in - m_out})
display(pd.DataFrame(r_contrib).round(2))

print('=== 항목별 한계기여 (k별, 포함−제외 t 차이) ===')
kv = res['k'].to_numpy()
r_kgrid = []
for j, c in enumerate(FVOL_COLS):
    row = {'item': c}
    for k in range(4, 14):
        mk = (kv == k) & ok_t
        a = tv[mk & inc[:, j]]
        b = tv[mk & ~inc[:, j]]
        row[f'k={k}'] = (a.mean() - b.mean()) if len(a) and len(b) else np.nan
    r_kgrid.append(row)
display(pd.DataFrame(r_kgrid).set_index('item').round(2))

print('=== 쌍 상호작용 (동시 제외 시너지, k 무조건부) ===')
pairs = []
for a in range(14):
    for b in range(a + 1, 14):
        m11 = tv[inc[:, a] & inc[:, b] & ok_t].mean()
        m01 = tv[~inc[:, a] & inc[:, b] & ok_t].mean()
        m10 = tv[inc[:, a] & ~inc[:, b] & ok_t].mean()
        m00 = tv[~inc[:, a] & ~inc[:, b] & ok_t].mean()
        syn = m00 - m01 - m10 + m11
        pairs.append({'pair': f'{FVOL_COLS[a]}+{FVOL_COLS[b]}',
                      'both_excl_t': m00, 'synergy': syn})
pr = pd.DataFrame(pairs)
print('동시 제외 시 평균 t 상위 8쌍:')
display(pr.nlargest(8, 'both_excl_t').round(3))
pr['abs_syn'] = pr['synergy'].abs()
print('시너지(비가법성) 절대값 상위 8쌍:')
display(pr.nlargest(8, 'abs_syn')[['pair', 'both_excl_t', 'synergy']].round(3))

print('=== Low/High 분해: t 상위 15 조합의 개선 원천 ===')
b14 = res[res['k'] == 14].iloc[0]
top = res.nlargest(15, 't').copy()
top['dLow'] = top['low'] - b14['low']
top['dHigh'] = top['high'] - b14['high']
print(f'(full14 기준 Low {b14["low"]:.2f} High {b14["high"]:.2f})')
display(top[['k', 'excluded', 'lh', 't', 'dLow', 'dHigh']].round(2))

조합 수: 16383
=== 정합성: full14 ===


,lh,t,low,high,n_form
16382,0.79,2.8,1.17,0.38,588.36


(기존 #8: LH 0.82 t 2.89 Low 1.18 High 0.36 / 형성 639)
=== k별 분포 ===


,lh_m,lh_sd,t_m,t_min,t_max,nform_m
k,,,,,,
1,0.64,0.22,2.57,0.55,3.85,572.36
2,0.71,0.21,2.69,0.63,4.22,556.23
3,0.72,0.19,2.64,0.52,4.00,540.57
4,0.79,0.14,2.93,0.87,4.33,586.62
5,0.79,0.13,2.91,1.44,4.20,585.48
6,0.79,0.12,2.87,1.55,4.15,584.15
7,0.78,0.11,2.84,1.55,4.03,587.86
8,0.77,0.09,2.79,1.91,3.96,587.52
9,0.77,0.08,2.75,2.01,3.97,587.11


=== k별 최고 조합 (t 기준) ===


...

,pair,both_excl_t,synergy
33,invtq+xoprq,3.192,0.105
34,invtq+cogsq,3.188,0.078
29,invtq+apq,3.130,0.059
28,invtq+dlcq,3.117,0.018
25,invtq+ppentq,3.096,0.051
88,xoprq+cogsq,3.060,0.146
27,invtq+lctq,3.055,0.054
30,invtq+dlttq,3.055,-0.017


시너지(비가법성) 절대값 상위 8쌍:


,pair,both_excl_t,synergy
75,apq+xsgaq,2.762,-0.162
88,xoprq+cogsq,3.060,0.146
38,ppentq+dlcq,2.920,0.140
12,actq+xsgaq,2.654,-0.118
24,rectq+xsgaq,2.708,-0.109
18,rectq+apq,2.826,-0.108
55,lctq+dlcq,2.870,0.107
33,invtq+xoprq,3.192,0.105


=== Low/High 분해: t 상위 15 조합의 개선 원천 ===
(full14 기준 Low 1.17 High 0.38)


,k,excluded,lh,t,dLow,dHigh
685,4,rectq|invtq|ppentq|atq|dlcq|apq|ltq|xoprq|cogs...,1.15,4.33,0.21,-0.15
29,2,actq|invtq|ppentq|lctq|dlcq|apq|dlttq|ltq|seqq...,1.12,4.22,0.24,-0.09
2025,5,rectq|invtq|ppentq|lctq|dlcq|ltq|xoprq|cogsq|x...,1.11,4.20,0.11,-0.21
2150,5,rectq|invtq|ppentq|atq|lctq|dlcq|xoprq|cogsq|x...,1.11,4.19,0.10,-0.22
3817,6,invtq|ppentq|lctq|dlcq|dlttq|ltq|xoprq|cogsq,1.14,4.15,0.17,-0.18
1330,4,actq|rectq|invtq|ppentq|lctq|dlcq|apq|ltq|xopr...,1.14,4.12,0.18,-0.17
3821,6,invtq|ppentq|lctq|dlcq|apq|xoprq|cogsq|xsgaq,1.11,4.10,0.14,-0.18
1383,4,actq|rectq|invtq|ppentq|atq|dlcq|apq|seqq|xopr...,1.10,4.09,0.12,-0.19
593,4,rectq|invtq|lctq|dlcq|apq|ltq|seqq|xoprq|cogsq...,1.00,4.09,0.17,-0.04
69,2,actq|rectq|invtq|ppentq|atq|apq|dlttq|ltq|seqq...,0.97,4.07,0.06,-0.13


In [14]:
# 13

prices = pd.read_parquet('prices.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

px = prices.copy()
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price', 'mktcap']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

f = sig.merge(form_px[['code', 'form_qtr', 'price', 'mktcap']], on=['code', 'form_qtr'], how='inner')
f = f[f['price'] >= 1000].dropna(subset=['mktcap']).copy()
f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)

print('=== 1. decile별 시총 점유율 (분기별 점유율의 평균, %) — 논문 Figure 2 대응 ===')
qsum = f.groupby('form_qtr')['mktcap'].transform('sum')
f['share'] = f['mktcap'] / qsum
share = f.groupby(['form_qtr', 'decile'])['share'].sum().reset_index()
avg_share = share.groupby('decile')['share'].mean() * 100
display(avg_share.round(1).rename_axis('decile').reset_index(name='시총점유%'))
print('(논문 미국: d1 16.3% → d10 5.2%)')

print('=== 2. decile별 시총 분포 (중앙값, 조원) ===')
med = f.groupby('decile')['mktcap'].median() / 1e12
display(med.round(3).rename_axis('decile').reset_index(name='시총중앙값_조원'))

print('=== 3. decile별 최대 종목 집중도 ===')
top1 = (f.sort_values('mktcap', ascending=False)
        .groupby(['form_qtr', 'decile']).head(1)
        .groupby(['form_qtr', 'decile'])['mktcap'].sum())
dsum = f.groupby(['form_qtr', 'decile'])['mktcap'].sum()
conc = (top1 / dsum).reset_index(name='c').groupby('decile')['c'].mean() * 100
print('decile 내 1위 종목의 시총 비중 평균(%):')
display(conc.round(1).rename_axis('decile').reset_index(name='1위종목_시총비중%'))

print('=== 4. d1·d10의 대표 얼굴 (최근 형성 분기) ===')
last_q = f['form_qtr'].max()
recent = f[f['form_qtr'] == last_q]
for dc in [1, 10]:
    top = recent[recent['decile'] == dc].nlargest(5, 'mktcap')
    print(f'decile {dc} 시총 상위 5 ({last_q}):')
    display(top[['code', 'mktcap']].assign(조원=lambda x: (x['mktcap']/1e12).round(2))[['code', '조원']])

print('=== 5. 시총 5분위 × FVOL: 대형주에서 신호가 있는가 ===')
f['size_q'] = f.groupby('form_qtr')['mktcap'].transform(
    lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
f['fq'] = f.groupby(['form_qtr', 'size_q'])['FVOL'].transform(
    lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
f = f.dropna(subset=['fq'])
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

px['ret'] = px['ret'] / 100.0
ret_m = px[['code', 'ym', 'ret']].dropna()
holds = []
for k in [1, 2, 3]:
    h = f[['code', 'form_ym', 'size_q', 'fq']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)
m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
m = m.drop_duplicates(['code', 'form_ym', 'ym'])
cell = m.groupby(['size_q', 'ym', 'fq'])['ret'].mean().reset_index(name='r')

print('사이즈 분위(1=소형, 5=대형)별 FVOL Low-High (EW, %/월):')
r_sz = []
for sq in range(1, 6):
    w = cell[cell['size_q'] == sq].pivot(index='ym', columns='fq', values='r')
    if 1 not in w or 5 not in w:
        continue
    lh = (w[1] - w[5]).dropna()
    t = lh.mean() / (lh.std(ddof=1) / np.sqrt(len(lh)))
    r_sz.append({'size_q': sq, 'L-H%': lh.mean()*100, 't': t, '월수': len(lh)})
display(pd.DataFrame(r_sz).round(2))

=== 1. decile별 시총 점유율 (분기별 점유율의 평균, %) — 논문 Figure 2 대응 ===


,decile,시총점유%
0,1,23.3
1,2,14.8
2,3,11.5
3,4,9.4
4,5,8.3
5,6,7.8
6,7,8.1
7,8,6.7
8,9,5.5
9,10,4.6


(논문 미국: d1 16.3% → d10 5.2%)
=== 2. decile별 시총 분포 (중앙값, 조원) ===


,decile,시총중앙값_조원
0,1,0.276
1,2,0.217
2,3,0.200
3,4,0.183
4,5,0.178
5,6,0.173
6,7,0.157
7,8,0.137
8,9,0.123
9,10,0.105


=== 3. decile별 최대 종목 집중도 ===
decile 내 1위 종목의 시총 비중 평균(%):


...

,code,조원
1466,A000270,53.88
37446,A012330,45.37
55570,A066570,33.07
21707,A005490,25.12
41305,A015760,23.75


decile 10 시총 상위 5 (2026Q2):


,code,조원
64749,A207940,64.39
37639,A012450,51.31
55831,A068270,40.30
24988,A006400,39.25
53039,A042700,24.45


=== 5. 시총 5분위 × FVOL: 대형주에서 신호가 있는가 ===
사이즈 분위(1=소형, 5=대형)별 FVOL Low-High (EW, %/월):


,size_q,L-H%,t,월수
0,1,0.86,2.00,291
1,2,1.58,5.08,291
2,3,0.60,1.92,291
3,4,0.47,1.77,291
4,5,0.30,1.05,291


In [15]:
# 14

financials = pd.read_parquet('financials.parquet')
prices = pd.read_parquet('prices.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')

form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f = f[f['price'] >= 1000].copy()
f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

holds = []
for k in [1, 2, 3]:
    h = f[['code', 'form_ym', 'decile']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

m = px[['code', 'ym', 'ret']].dropna().merge(
    hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
m = m.drop_duplicates(['code', 'form_ym', 'ym'])

ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='r')
wide = ew.pivot(index='ym', columns='decile', values='r')
lh_m = (wide[1] - wide[10]).dropna()

yr = lh_m.copy()
yr.index = yr.index.year
lo_y = wide[1].dropna().copy(); lo_y.index = lo_y.index.year
hi_y = wide[10].dropna().copy(); hi_y.index = hi_y.index.year
lo_ann = lo_y.groupby(level=0).mean()
hi_ann = hi_y.groupby(level=0).mean()

r_yr = []
for y, g in yr.groupby(level=0):
    win = '승' if g.mean() > 0 else '패'
    pos = (g > 0).mean()
    r_yr.append({'연도': y, '격차%/월': g.mean()*100, '승패': win,
                 '양수월%': pos*100, 'Low%': lo_ann[y]*100, 'High%': hi_ann[y]*100})
display(pd.DataFrame(r_yr).round({'격차%/월': 2, '양수월%': 0, 'Low%': 2, 'High%': 2}))

ann = yr.groupby(level=0).mean()
print(f'승리 연도: {(ann > 0).sum()}/{len(ann)}  '
      f'최고 {ann.idxmax()} ({ann.max()*100:.2f}%/월)  '
      f'최악 {ann.idxmin()} ({ann.min()*100:.2f}%/월)')

,연도,격차%/월,승패,양수월%,Low%,High%
0,2002,3.74,승,67.0,-2.29,-6.02
1,2003,3.05,승,67.0,2.05,-1.01
2,2004,1.05,승,67.0,1.46,0.41
3,2005,-1.85,패,50.0,5.81,7.66
4,2006,2.36,승,75.0,1.19,-1.17
5,2007,1.34,승,58.0,2.58,1.23
6,2008,3.16,승,58.0,-2.94,-6.09
7,2009,0.53,승,58.0,3.79,3.27
8,2010,1.22,승,75.0,1.87,0.65
9,2011,1.18,승,58.0,0.32,-0.86


승리 연도: 18/25  최고 2002 (3.74%/월)  최악 2005 (-1.85%/월)


In [16]:
# 15

raw = pd.read_excel(XLSX, sheet_name='33_industry', header=None)

codes = raw.iloc[7, 1:].astype(str).tolist()
body = raw.iloc[14:, :].copy()
dates = pd.PeriodIndex(body.iloc[:, 0].astype(str), freq='M')

ind_wide = body.iloc[:, 1:]
ind_wide.columns = codes
ind_wide.index = dates.year

ind_wide = ind_wide.rename_axis(index='year', columns='code')
ind_long = ind_wide.stack().rename('industry').reset_index()
ind_long = ind_long.dropna(subset=['industry'])
ind_long['industry'] = ind_long['industry'].astype(str).str.strip()
ind_long = ind_long[ind_long['industry'].ne('') & ind_long['industry'].ne('nan')]
ind_long = ind_long[~ind_long['code'].isin(EXCLUDE_FIN)]

ind_long.to_parquet('industry.parquet')
print(f'industry.parquet 저장: {len(ind_long)}행, 종목 {ind_long["code"].nunique()}개, '
      f'연도 {ind_long["year"].min()}~{ind_long["year"].max()}')
print(f'대분류 종류({ind_long["industry"].nunique()}개): {sorted(ind_long["industry"].unique())}')

financials = pd.read_parquet('financials.parquet')
fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

valid = fv[fv['FVOL'].notna()][['code', 'date']].copy()
valid['apply_year'] = valid['date'].dt.year
valid['ind_year'] = valid['apply_year'] - 1

merged = valid.merge(ind_long.rename(columns={'year': 'ind_year'}),
                     on=['code', 'ind_year'], how='left')
miss = merged['industry'].isna()
print(f'FVOL 유효 종목-분기 {len(merged)}개 중 분류 결측 {miss.sum()}개 ({miss.mean():.2%})')
if miss.any():
    mc = merged[miss]
    print(f'결측 종목 수: {mc["code"].nunique()}개, 결측 집중 연도 상위 5:')
    display(mc['apply_year'].value_counts().head(5)
            .rename_axis('연도').reset_index(name='결측수'))

print('연도별 대분류 분포 (종목 수, FVOL 유효 기준, 5년 간격):')
tab = merged.dropna(subset=['industry'])
pv = tab[tab['apply_year'].isin([2005, 2010, 2015, 2020, 2025])].pivot_table(
    index='industry', columns='apply_year', values='code', aggfunc='nunique', fill_value=0)
display(pv)

industry.parquet 저장: 18387행, 종목 995개, 연도 1999~2025
대분류 종류(9개): ['IT', '건강관리', '경기관련소비재', '산업재', '소재', '에너지', '유틸리티', '커뮤니케이션서비스', '필수소비재']
FVOL 유효 종목-분기 66765개 중 분류 결측 1086개 (1.63%)
결측 종목 수: 137개, 결측 집중 연도 상위 5:


,연도,결측수
0,2006,97
1,2005,88
2,2007,87
3,2004,81
4,2008,81


연도별 대분류 분포 (종목 수, FVOL 유효 기준, 5년 간격):


apply_year,2005,2010,2015,2020,2025
industry,,,,,
IT,76,78,66,66,75
건강관리,42,44,48,52,61
경기관련소비재,141,150,160,169,175
산업재,137,144,163,173,179
소재,141,153,153,149,155
에너지,7,11,13,13,17
유틸리티,12,14,17,17,16
커뮤니케이션서비스,5,5,3,27,26
필수소비재,50,51,56,58,55


In [17]:
# 16

financials = pd.read_parquet('financials.parquet')
ind_long = pd.read_parquet('industry.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

v = fv[fv['FVOL'].notna()][['code', 'date', 'FVOL']].copy()
v['ind_year'] = v['date'].dt.year - 1
v = v.merge(ind_long.rename(columns={'year': 'ind_year'}),
            on=['code', 'ind_year'], how='inner')

q = v.groupby(['date', 'industry'])['FVOL'].agg(['mean', 'count']).reset_index()
lvl = q.groupby('industry').apply(
    lambda g: pd.Series({'평균FVOL': g['mean'].mean(),
                         '분기당종목': g['count'].mean()}), include_groups=False)
lvl = lvl.sort_values('평균FVOL', ascending=False)
print('=== 1. 업종별 평균 FVOL rank (0~1, 높을수록 고변동) ===')
display(lvl.round(3))

grand = v['FVOL'].mean()
between = v.groupby('industry')['FVOL'].agg(['mean', 'count'])
ss_between = (between['count'] * (between['mean'] - grand) ** 2).sum()
ss_total = ((v['FVOL'] - grand) ** 2).sum()
print(f'=== 2. FVOL 분산 중 업종 간 차이가 설명하는 비율 ===')
print(f'업종 간 분산 비율: {ss_between / ss_total:.1%}  (나머지 {1 - ss_between/ss_total:.1%}는 업종 내 기업 간 차이)')

v['decile'] = v.groupby('date')['FVOL'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop')) + 1
base = v.groupby('industry').size() / len(v)
print('=== 3. 업종 비중: 전체 vs d1 vs d10 (%) ===')
out = pd.DataFrame({'전체': base * 100})
for dc, nm in [(1, 'd1(저변동)'), (10, 'd10(고변동)')]:
    d = v[v['decile'] == dc]
    out[nm] = d.groupby('industry').size() / len(d) * 100
out['d10−전체'] = out['d10(고변동)'] - out['전체']
display(out.round(1).sort_values('d10−전체', ascending=False))

=== 1. 업종별 평균 FVOL rank (0~1, 높을수록 고변동) ===


,평균FVOL,분기당종목
industry,,
IT,0.593,71.245
산업재,0.521,156.510
에너지,0.515,10.980
건강관리,0.512,48.102
경기관련소비재,0.494,158.459
소재,0.478,146.806
커뮤니케이션서비스,0.472,9.776
필수소비재,0.430,53.194
유틸리티,0.334,15.122


=== 2. FVOL 분산 중 업종 간 차이가 설명하는 비율 ===
업종 간 분산 비율: 5.3%  (나머지 94.7%는 업종 내 기업 간 차이)
=== 3. 업종 비중: 전체 vs d1 vs d10 (%) ===


,전체,d1(저변동),d10(고변동),d10−전체
industry,,,,
IT,10.6,4.4,20.5,9.8
산업재,23.4,20.3,25.7,2.3
에너지,1.6,2.4,3.3,1.7
건강관리,7.2,5.1,8.7,1.5
경기관련소비재,23.6,25.4,24.2,0.6
커뮤니케이션서비스,1.5,0.5,2.0,0.5
유틸리티,2.3,7.5,0.3,-2.0
필수소비재,7.9,15.1,4.3,-3.6
소재,21.9,19.2,11.1,-10.8


In [18]:
# 17

financials = pd.read_parquet('financials.parquet')
prices = pd.read_parquet('prices.parquet')
ind_long = pd.read_parquet('industry.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['ind_year'] = sig['date'].dt.year - 1
sig = sig.merge(ind_long.rename(columns={'year': 'ind_year'}),
                on=['code', 'ind_year'], how='inner')
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')

form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f = f[f['price'] >= 1000].copy()
f['q5'] = f.groupby(['form_qtr', 'industry'])['FVOL'].transform(
    lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
f = f.dropna(subset=['q5'])
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

ret_m = px[['code', 'ym', 'ret']].dropna()
holds = []
for k in [1, 2, 3]:
    h = f[['code', 'form_ym', 'industry', 'q5']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
m = m.drop_duplicates(['code', 'form_ym', 'ym'])
cell = m.groupby(['industry', 'ym', 'q5'])['ret'].mean().reset_index(name='r')

nq = f.groupby(['form_qtr', 'industry']).size().groupby('industry').mean()

r_ind = []
for indus in sorted(f['industry'].unique()):
    w = cell[cell['industry'] == indus].pivot(index='ym', columns='q5', values='r')
    if 1 not in w.columns or 5 not in w.columns:
        continue
    lh = (w[1] - w[5]).dropna()
    if len(lh) < 24:
        continue
    t = lh.mean() / (lh.std(ddof=1) / np.sqrt(len(lh)))
    r_ind.append({'industry': indus, 'L-H%': lh.mean()*100, 't': t,
                  'Low%': w[1].mean()*100, 'High%': w[5].mean()*100,
                  '월수': len(lh), '종목/분기': nq[indus]})
display(pd.DataFrame(r_ind).round({'L-H%': 2, 't': 2, 'Low%': 2, 'High%': 2, '종목/분기': 0}))

,industry,L-H%,t,Low%,High%,월수,종목/분기
0,IT,1.14,2.22,1.64,0.49,291,66.0
1,건강관리,0.21,0.49,1.13,0.92,291,47.0
2,경기관련소비재,0.73,2.40,1.01,0.28,291,147.0
3,산업재,0.58,1.76,1.37,0.79,291,148.0
4,소재,0.47,1.83,1.31,0.84,291,136.0
5,에너지,1.16,1.71,1.39,0.23,291,10.0
6,유틸리티,-0.08,-0.20,1.23,1.31,291,15.0
7,커뮤니케이션서비스,0.52,0.92,0.34,-0.11,288,10.0
8,필수소비재,0.15,0.33,1.14,0.99,291,51.0


In [19]:
# 18

financials = pd.read_parquet('financials.parquet')

DEL_BAD = ['감사의견', '정리절차', '자본잠식']
bad_set = set(delisted[delisted['reason_cat'].isin(DEL_BAD)]['code'])
mrg_set = set(delisted[delisted['reason_cat'] == '합병·해산']['code'])
del_q = {}
for _, r in delisted.iterrows():
    if pd.notna(r['del_date']):
        pq = pd.Period(r['del_date'], freq='Q')
        del_q[r['code']] = pq.year * 4 + pq.quarter - 1

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

d = fv[fv['FVOL'].notna()][['code', 'date', 'FVOL', 'ibq']].copy()
pq = pd.PeriodIndex(d['date'], freq='Q')
d['qidx'] = pq.year * 4 + (pq.quarter - 1)
d['dec'] = d.groupby('date')['FVOL'].transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop')) + 1

def within8(code, qidx, target_set):
    dq = del_q.get(code)
    return int(dq is not None and code in target_set and 0 < dq - qidx <= 8)

d['bad8'] = [within8(c, q, bad_set) for c, q in zip(d['code'], d['qidx'])]
d['mrg8'] = [within8(c, q, mrg_set) for c, q in zip(d['code'], d['qidx'])]

print(f'FVOL 관측치 {len(d)}개, 종목 {d["code"].nunique()}개')
print(f'전체 8분기내 부실폐지율 {d["bad8"].mean():.2%}  합병폐지율 {d["mrg8"].mean():.2%}')

print('=== A. decile별 8분기내 폐지율 (%) ===')
r_A = []
for dc in range(1, 11):
    s = d[d['dec'] == dc]
    r_A.append({'dec': dc, '부실%': s['bad8'].mean()*100, '합병%': s['mrg8'].mean()*100})
display(pd.DataFrame(r_A).round(2))
base = d['bad8'].mean()
print(f'd10 부실폐지율 / 전체 = {d[d["dec"]==10]["bad8"].mean()/base:.1f}배')

print('=== B. 흑자 한정(ibq>0) decile별 부실폐지율 (%) ===')
dp = d[d['ibq'] > 0]
print(f'흑자 관측치 {len(dp)}개')
r_B = []
for dc in range(1, 11):
    s = dp[dp['dec'] == dc]
    r_B.append({'dec': dc, '부실%': s['bad8'].mean()*100})
display(pd.DataFrame(r_B).round(2))
print(f'흑자 전체 부실폐지율 {dp["bad8"].mean():.2%}  d10 {dp[dp["dec"]==10]["bad8"].mean():.2%}')

print('=== C. 폐지 -12분기 FVOL rank 궤적 (부실 폐지 vs 생존) ===')
surv_mean = d[~d['code'].isin(bad_set)]['FVOL'].mean()
r_C = []
for off in range(-12, 1):
    vals = []
    for c in bad_set:
        dq = del_q.get(c)
        if dq is None:
            continue
        sub = d[(d['code'] == c) & (d['qidx'] == dq + off)]
        if len(sub):
            vals.append(sub['FVOL'].iloc[0])
    m = np.mean(vals) if vals else np.nan
    r_C.append({'offset': off, '부실폐지': m, '생존': surv_mean})
display(pd.DataFrame(r_C).round(3))

FVOL 관측치 66765개, 종목 951개
전체 8분기내 부실폐지율 0.94%  합병폐지율 0.69%
=== A. decile별 8분기내 폐지율 (%) ===


,dec,부실%,합병%
0,1,0.03,0.49
1,2,0.03,0.75
2,3,0.08,0.75
3,4,0.06,0.43
4,5,0.10,0.72
5,6,0.26,0.62
6,7,0.24,0.93
7,8,0.61,0.90
8,9,1.86,0.74
9,10,6.11,0.54


d10 부실폐지율 / 전체 = 6.5배
=== B. 흑자 한정(ibq>0) decile별 부실폐지율 (%) ===
흑자 관측치 47260개


,dec,부실%
0,1,0.00
1,2,0.00
2,3,0.02
3,4,0.02
4,5,0.02
5,6,0.04
6,7,0.02
7,8,0.07
8,9,0.82
9,10,1.58


흑자 전체 부실폐지율 0.18%  d10 1.58%
=== C. 폐지 -12분기 FVOL rank 궤적 (부실 폐지 vs 생존) ===


,offset,부실폐지,생존
0,-12,0.770,0.491
1,-11,0.784,0.491
2,-10,0.795,0.491
3,-9,0.803,0.491
4,-8,0.801,0.491
5,-7,0.808,0.491
6,-6,0.819,0.491
7,-5,0.829,0.491
8,-4,0.840,0.491
9,-3,0.845,0.491


In [20]:
# 19

financials = pd.read_parquet('financials.parquet')
prices = pd.read_parquet('prices.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')

del_bad = delisted[delisted['reason_cat'].isin(['감사의견', '정리절차', '자본잠식'])]['code']
pdd = px[px['code'].isin(del_bad) & px['ret'].notna()].sort_values(['code', 'date']).copy()
n_bad = pdd['code'].nunique()

print(f'[F8] 부실 폐지 종목(수익률 존재): {n_bad}개')
print('폐지 직전 누적수익률 (마지막 k개월, %)')
r_cum = []
for k in [1, 3, 6, 12]:
    cum = pdd.groupby('code').tail(k).groupby('code')['ret'].apply(lambda s: (1 + s).prod() - 1) * 100
    r_cum.append({'기간': f'{k}개월', '평균': cum.mean(), '중앙값': cum.median(), 'n': len(cum)})
display(pd.DataFrame(r_cum).round(1))

print('폐지 전 월별 수익률 궤적 (offset 0 = 실측 마지막 달, %)')
pdd['off'] = pdd.groupby('code').cumcount(ascending=False)
r_traj = []
for o in range(0, 13):
    s = pdd[pdd['off'] == o]['ret'] * 100
    r_traj.append({'offset': -o, '평균': s.mean(), '중앙값': s.median(), 'n': len(s)})
display(pd.DataFrame(r_traj).round(1))

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1
form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')
f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f = f[f['price'] >= 1000].copy()
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

holds = []
for k in [1, 2, 3]:
    h = f[['code', 'form_ym']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

held = px[['code', 'ym', 'ret']].dropna().merge(
    hold[['code', 'hold_ym']].drop_duplicates(),
    left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
held = held.drop_duplicates(['code', 'ym']).sort_values(['code', 'ym'])

n_obs = len(held)
n_zero = (held['ret'] == 0).sum()

held['is0'] = (held['ret'] == 0).astype(int)
def max_run(s):
    m = c = 0
    for v in s:
        c = c + 1 if v == 1 else 0
        m = max(m, c)
    return m
runs = held.groupby('code')['is0'].apply(max_run)
codes_3plus = set(runs[runs >= 3].index)

held['grp'] = (held['is0'] != held.groupby('code')['is0'].shift()).cumsum()
runlen = held[held['is0'] == 1].groupby(['code', 'grp']).size()
months_in_3plus = runlen[runlen >= 3].sum()

print('[F9] 실제 매매 대상 종목-월의 정지(ret==0) 실태')
r_stop = [{
    '전체 종목-월': n_obs,
    'ret==0 월수': int(n_zero),
    'ret==0 비율%': n_zero / n_obs * 100,
    '3개월+연속정지에 속한 월수': int(months_in_3plus),
    '3개월+연속정지 월 비율%': months_in_3plus / n_obs * 100,
}]
display(pd.DataFrame(r_stop).round(2))

[F8] 부실 폐지 종목(수익률 존재): 105개
폐지 직전 누적수익률 (마지막 k개월, %)


,기간,평균,중앙값,n
0,1개월,-28.9,-4.9,105
1,3개월,-32.1,-33.3,105
2,6개월,-41.5,-49.7,105
3,12개월,-51.1,-69.1,105


폐지 전 월별 수익률 궤적 (offset 0 = 실측 마지막 달, %)


,offset,평균,중앙값,n
0,0,-28.9,-5.0,105
1,-1,-7.2,0.0,105
2,-2,4.8,0.0,105
3,-3,-7.7,-7.0,105
4,-4,-1.1,0.0,105
5,-5,2.1,0.0,105
6,-6,-8.1,-5.5,101
7,-7,-7.0,-2.0,101
8,-8,-7.9,-6.4,101
9,-9,5.5,0.0,100


[F9] 실제 매매 대상 종목-월의 정지(ret==0) 실태


,전체 종목-월,ret==0 월수,ret==0 비율%,3개월+연속정지에 속한 월수,3개월+연속정지 월 비율%
0,182868,3173,1.74,1302,0.71


In [21]:
# 20

financials = pd.read_parquet('financials.parquet')
prices = pd.read_parquet('prices.parquet')
market = pd.read_parquet('market.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f = f[f['price'] >= 1000].copy()
f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

holds = []
for k in [1, 2, 3]:
    h = f[['code', 'form_ym', 'decile']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

m = px[['code', 'ym', 'ret']].dropna().merge(
    hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
m = m.drop_duplicates(['code', 'form_ym', 'ym'])
ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='r')
wide = ew.pivot(index='ym', columns='decile', values='r')
lh = (wide[1] - wide[10]).dropna()

mk = market.dropna(subset=['kret']).copy()
mk['kret'] = mk['kret'] / 100.0
mk['ym'] = pd.PeriodIndex(mk['date'], freq='M')

ax_rows = []
for t in lh.index:
    sub = mk[mk['ym'].isin([t - 2, t - 1, t])]['kret']
    if len(sub) < 20:
        continue
    ax_rows.append({'ym': t, 'vol': sub.std(), 'trend': (1 + sub).prod() - 1, 'lh': lh[t]})
regime = pd.DataFrame(ax_rows).dropna()

vol_med = regime['vol'].median()
regime['quad'] = ((regime['vol'] > vol_med).map({True: '고변동', False: '저변동'})
                  + (regime['trend'] > 0).map({True: '상승', False: '하락'}))
regime['lh_pct'] = regime['lh'] * 100

qstat = regime.groupby('quad').agg(
    n=('lh', 'size'), vol=('vol', 'mean'), trend=('trend', 'mean'),
    lh_mean=('lh', 'mean'),
    lh_t=('lh', lambda s: s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))))
print(f'월 {len(regime)}개, 변동성 중앙값 {vol_med:.4f}')
r_qs = []
for q in qstat.sort_values('lh_mean', ascending=False).index:
    r = qstat.loc[q]
    r_qs.append({'국면': q, 'n': int(r['n']), '변동성': r['vol'], '추세': r['trend'],
                 'LH%': r['lh_mean']*100, 't': r['lh_t']})
display(pd.DataFrame(r_qs).round({'변동성': 4, '추세': 4, 'LH%': 2, 't': 2}))

print('=== 국면 더미 회귀 (기준: 저변동하락, robust SE HC3) ===')
regime['quad_c'] = pd.Categorical(
    regime['quad'], categories=['저변동하락', '저변동상승', '고변동상승', '고변동하락'])
mod = smf.ols('lh_pct ~ C(quad_c, Treatment(reference="저변동하락"))', data=regime).fit(cov_type='HC3')
print(mod.summary().tables[1])
print('계수 = 각 국면이 저변동하락 대비 Low-High(%/월)가 얼마나 높은가')

print('=== 극단 쌍 2표본 검정 (Welch, 이분산 미가정) ===')
g_hi = regime[regime['quad'] == '고변동하락']['lh_pct']
g_lo = regime[regime['quad'] == '저변동하락']['lh_pct']
tstat, pval = stats.ttest_ind(g_hi, g_lo, equal_var=False)
print(f'고변동하락 (n={len(g_hi)}, 평균 {g_hi.mean():.2f}%) vs '
      f'저변동하락 (n={len(g_lo)}, 평균 {g_lo.mean():.2f}%)')
print(f'평균 차이 {g_hi.mean() - g_lo.mean():.2f}%p   Welch t={tstat:.2f}  p={pval:.4f}')

월 291개, 변동성 중앙값 0.0109


,국면,n,변동성,추세,LH%,t
0,고변동하락,61,0.0176,-0.0927,2.42,3.30
1,저변동상승,100,0.0082,0.0620,0.68,1.90
2,고변동상승,84,0.0158,0.1275,0.48,0.83
3,저변동하락,46,0.0086,-0.0419,-0.58,-0.90


=== 국면 더미 회귀 (기준: 저변동하락, robust SE HC3) ===
                                                       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------
Intercept                                           -0.5775      0.648     -0.891      0.373      -1.847       0.692
C(quad_c, Treatment(reference="저변동하락"))[T.저변동상승]     1.2619      0.742      1.700      0.089      -0.193       2.717
C(quad_c, Treatment(reference="저변동하락"))[T.고변동상승]     1.0585      0.871      1.215      0.224      -0.649       2.766
C(quad_c, Treatment(reference="저변동하락"))[T.고변동하락]     2.9956      0.983      3.047      0.002       1.069       4.923
계수 = 각 국면이 저변동하락 대비 Low-High(%/월)가 얼마나 높은가
=== 극단 쌍 2표본 검정 (Welch, 이분산 미가정) ===
고변동하락 (n=61, 평균 2.42%) vs 저변동하락 (n=46, 평균 -0.58%)
평균 차이 3.00%p   Welch t=3.08  p=0.0027


In [22]:
# 21

financials = pd.read_parquet('financials.parquet')
prices = pd.read_parquet('prices.parquet')
market = pd.read_parquet('market.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f = f[f['price'] >= 1000].copy()
f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

holds = []
for k in [1, 2, 3]:
    h = f[['code', 'form_ym', 'decile']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

m = px[['code', 'ym', 'ret']].dropna().merge(
    hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
m = m.drop_duplicates(['code', 'form_ym', 'ym'])
ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='r')
wide = ew.pivot(index='ym', columns='decile', values='r')
lh = (wide[1] - wide[10]).dropna()

mk = market.dropna(subset=['kret']).copy()
mk['kret'] = mk['kret'] / 100.0
mk['ym'] = pd.PeriodIndex(mk['date'], freq='M')

rows = []
for t in lh.index:
    sub = mk[mk['ym'].isin([t - 2, t - 1, t])]['kret']
    if len(sub) < 20:
        continue
    rows.append({'ym': t, 'vol': sub.std(), 'trend': (1 + sub).prod() - 1, 'lh': lh[t]})
regime = pd.DataFrame(rows).dropna()

vol_med = regime['vol'].median()
regime['quad'] = ((regime['vol'] > vol_med).map({True: '고변동', False: '저변동'})
                  + (regime['trend'] > 0).map({True: '상승', False: '하락'}))
regime['year'] = regime['ym'].dt.year
regime['lh_pct'] = regime['lh'] * 100

QUADS = ['고변동하락', '저변동상승', '고변동상승', '저변동하락']

yr = regime.groupby('year').agg(
    n=('lh', 'size'), lh=('lh_pct', 'mean'),
    pos=('lh_pct', lambda s: (s > 0).mean()))
yr['승패'] = np.where(yr['lh'] > 0, '승', '패')
comp = regime.pivot_table(index='year', columns='quad', values='lh', aggfunc='size', fill_value=0)
for q in QUADS:
    if q not in comp.columns:
        comp[q] = 0
comp_pct = comp[QUADS].div(comp[QUADS].sum(axis=1), axis=0) * 100

print('=== 1. 연도별 승패 + 레짐 구성 비율 (관측월 분모, %) ===')
tbl1 = pd.DataFrame({
    '월': yr['n'].astype(int),
    '격차': yr['lh'].round(2),
    '양수월': (yr['pos'] * 100).round(0).astype(int),
    '승패': yr['승패'],
})
for q in QUADS:
    tbl1[q] = comp_pct[q].round(0).astype(int)
tbl1.index.name = '연도'
display(tbl1)

print('=== 2. 승 연도 vs 패 연도 평균 레짐 구성 (%) ===')
win_yrs = yr[yr['승패'] == '승'].index
los_yrs = yr[yr['승패'] == '패'].index
print(f'승 연도 {len(win_yrs)}개, 패 연도 {len(los_yrs)}개')
win_mean = comp_pct.loc[win_yrs, QUADS].mean()
los_mean = comp_pct.loc[los_yrs, QUADS].mean()
tbl2 = pd.DataFrame({
    '승연도평균': win_mean.round(1),
    '패연도평균': los_mean.round(1),
    '차이': (win_mean - los_mean).round(1),
})
tbl2.index.name = '국면'
display(tbl2)

print('=== 3. 국면별 양수월 비율·평균격차 (월 단위, %) ===')
tbl3 = pd.DataFrame({
    'n': [len(regime[regime['quad'] == q]) for q in QUADS],
    '양수월%': [round((regime[regime['quad'] == q]['lh'] > 0).mean() * 100) for q in QUADS],
    '평균격차%': [round(regime[regime['quad'] == q]['lh_pct'].mean(), 2) for q in QUADS],
}, index=pd.Index(QUADS, name='국면'))
display(tbl3)


=== 1. 연도별 승패 + 레짐 구성 비율 (관측월 분모, %) ===


,월,격차,양수월,승패,고변동하락,저변동상승,고변동상승,저변동하락
연도,,,,,,,,
2002,9,3.74,67,승,89,0,11,0
2003,12,3.05,67,승,25,0,75,0
2004,12,1.05,67,승,33,8,58,0
2005,12,-1.85,50,패,8,58,25,8
2006,12,2.36,75,승,33,33,33,0
2007,12,1.34,58,승,8,33,42,17
2008,12,3.16,58,승,83,0,17,0
2009,12,0.53,58,승,17,0,83,0
2010,12,1.22,75,승,0,67,33,0


=== 2. 승 연도 vs 패 연도 평균 레짐 구성 (%) ===
승 연도 18개, 패 연도 7개


,승연도평균,패연도평균,차이
국면,,,
고변동하락,25.8,9.5,16.2
저변동상승,30.1,41.7,-11.6
고변동상승,34.0,20.2,13.7
저변동하락,10.2,28.6,-18.4


=== 3. 국면별 양수월 비율·평균격차 (월 단위, %) ===


,n,양수월%,평균격차%
국면,,,
고변동하락,61,66,2.42
저변동상승,100,60,0.68
고변동상승,84,58,0.48
저변동하락,46,43,-0.58


In [23]:
# 22

financials = pd.read_parquet('financials.parquet')
prices = pd.read_parquet('prices.parquet')
market = pd.read_parquet('market.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')
ret_m = px[['code', 'ym', 'ret']].dropna()

mk = market.dropna(subset=['kret']).copy()
mk['kret'] = mk['kret'] / 100.0
mk['ym'] = pd.PeriodIndex(mk['date'], freq='M')
rows = []
for t in sorted(px['ym'].unique()):
    sub = mk[mk['ym'].isin([t - 2, t - 1, t])]['kret']
    if len(sub) >= 20:
        rows.append({'ym': t, 'vol': sub.std(), 'trend': (1 + sub).prod() - 1})
regime = pd.DataFrame(rows)
vol_med = regime['vol'].median()
regime['quad'] = ((regime['vol'] > vol_med).map({True: '고변동', False: '저변동'})
                  + (regime['trend'] > 0).map({True: '상승', False: '하락'}))
QUADS = ['고변동하락', '저변동상승', '고변동상승', '저변동하락']

def lh_of(valcol):
    s = fv[['code', 'date', valcol]].dropna().copy()
    s['form_qtr'] = pd.PeriodIndex(s['date'], freq='Q') + 1
    f = s.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
    f = f[f['price'] >= 1000].copy()
    f['decile'] = np.ceil(f.groupby('form_qtr')[valcol].rank(pct=True) * 10).clip(1, 10).astype(int)
    f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')
    holds = []
    for k in [1, 2, 3]:
        h = f[['code', 'form_ym', 'decile']].copy()
        h['hold_ym'] = h['form_ym'] + k
        holds.append(h)
    hold = pd.concat(holds, ignore_index=True)
    m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
    m = m.drop_duplicates(['code', 'form_ym', 'ym'])
    ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='r')
    w = ew.pivot(index='ym', columns='decile', values='r')
    return (w[1] - w[10]).dropna()

FVOL_LH = {c: lh_of(f'fvol_{c}') for c in FVOL_COLS}

print('[1-A] 항목별 × 국면 Low-High 평균 (%)')
r1a = []
for c in FVOL_COLS:
    d = FVOL_LH[c].rename('lh').reset_index().merge(regime[['ym', 'quad']], on='ym').dropna()
    row = {'항목': c}
    for q in QUADS:
        row[q] = d[d['quad'] == q]['lh'].mean() * 100
    r1a.append(row)
display(pd.DataFrame(r1a).round(2))

print('[1-B] 항목별 무조건부 Low-High (전기간) (%)')
r1b = []
for c in FVOL_COLS:
    lh = FVOL_LH[c]
    t = lh.mean() / (lh.std(ddof=1) / np.sqrt(len(lh)))
    r1b.append({'항목': c, 'LH%': lh.mean() * 100, 't': t})
display(pd.DataFrame(r1b).round(2))

print('[2-C] 항목별 결측률 (%)')
r2c = []
for c in [x for x in FVOL_COLS if x != 'xoprq']:
    r2c.append({'항목': c, '결측률%': financials[c].isna().mean() * 100})
display(pd.DataFrame(r2c).round(1))

[1-A] 항목별 × 국면 Low-High 평균 (%)


,항목,고변동하락,저변동상승,고변동상승,저변동하락
0,actq,2.13,0.68,1.02,-0.50
1,rectq,2.00,0.85,0.53,-0.88
2,invtq,0.82,-0.33,0.50,-0.18
3,ppentq,1.50,0.29,0.48,-0.46
4,atq,2.23,0.95,0.73,-0.58
5,lctq,1.39,1.13,1.01,-0.12
6,dlcq,1.61,0.72,0.71,-0.46
7,apq,1.57,0.66,0.47,-0.83
8,dlttq,1.00,0.54,0.28,-0.10
9,ltq,1.90,0.70,0.97,-0.47


[1-B] 항목별 무조건부 Low-High (전기간) (%)


,항목,LH%,t
0,actq,0.85,3.39
1,rectq,0.70,2.64
2,invtq,0.14,0.55
3,ppentq,0.45,2.30
4,atq,0.88,3.67
5,lctq,0.94,3.85
6,dlcq,0.69,2.97
7,apq,0.53,2.04
8,dlttq,0.45,1.74
9,ltq,0.81,3.12


[2-C] 항목별 결측률 (%)


,항목,결측률%
0,actq,36.2
1,rectq,36.2
2,invtq,38.2
3,ppentq,36.2
4,atq,36.2
5,lctq,36.2
6,dlcq,36.2
7,apq,36.2
8,dlttq,54.9
9,ltq,36.2


In [24]:
# 23

daily = pd.read_parquet('daily_returns.parquet')
market = pd.read_parquet('market.parquet')

IVOL_OUT = 'ivol_quarterly.parquet'
MIN_DAYS = 15
DRET_CAP = 100.0

if Path(IVOL_OUT).exists():
    ivol = pd.read_parquet(IVOL_OUT)
    print(f'{IVOL_OUT} loaded. rows {len(ivol)}  종목 {ivol["code"].nunique()}')
    print(f'분기당 평균 종목 {ivol.groupby("qtr").size().mean():.0f}')
    print(ivol['ivol'].describe()[['mean','std','min','50%','max']].round(4).to_dict())
else:
    mk = market.copy()
    mk['date'] = pd.to_datetime(mk['date'])
    mk['rf_d'] = mk['rf'] / 100 / 252
    mk['mkt_ex'] = mk['kret'] / 100 - mk['rf_d']
    mk = mk.dropna(subset=['mkt_ex', 'rf_d'])[['date', 'mkt_ex', 'rf_d']]

    d = daily.copy()
    d['date'] = pd.to_datetime(d['date'])
    d['dret'] = pd.to_numeric(d['dret'], errors='coerce')
    d = d.dropna(subset=['dret'])
    n_cut = (d['dret'].abs() > DRET_CAP).sum()
    d = d[d['dret'].abs() <= DRET_CAP]
    print(f'dret |>{DRET_CAP:.0f}%| 제거: {n_cut}건')

    d = d.merge(mk, on='date', how='inner')
    d['y'] = d['dret'] / 100 - d['rf_d']
    d = d.dropna(subset=['y', 'mkt_ex'])

    qtr = pd.PeriodIndex(d['date'], freq='Q')
    key = d['code'].astype(str).values + '_' + qtr.astype(str)
    uniq, gid = np.unique(key, return_inverse=True)
    G = len(uniq)

    y = d['y'].values.astype('float64')
    x = d['mkt_ex'].values.astype('float64')

    n   = np.bincount(gid, minlength=G).astype('float64')
    sx  = np.bincount(gid, weights=x,     minlength=G)
    sxx = np.bincount(gid, weights=x * x, minlength=G)
    sy  = np.bincount(gid, weights=y,     minlength=G)
    sxy = np.bincount(gid, weights=x * y, minlength=G)
    syy = np.bincount(gid, weights=y * y, minlength=G)

    det = n * sxx - sx * sx
    
    with np.errstate(divide='ignore', invalid='ignore'):
        b0 = (sxx * sy - sx * sxy) / det
        b1 = (n * sxy - sx * sy) / det
        ssr = syy - b0 * sy - b1 * sxy
        iv_raw = np.sqrt(ssr / (n - 2))
    mask = (n >= MIN_DAYS) & (det > 0) & (ssr > 0)
    ivol_arr = np.where(mask, iv_raw, np.nan)

    codes = np.array([u.rsplit('_', 1)[0] for u in uniq])
    qs    = np.array([u.rsplit('_', 1)[1] for u in uniq])
    ivol = pd.DataFrame({'code': codes,
                         'qtr': pd.PeriodIndex(qs, freq='Q'),
                         'ivol': ivol_arr}).dropna(subset=['ivol'])

    print(f'{IVOL_OUT} written. rows {len(ivol)}  종목 {ivol["code"].nunique()}')
    print(f'분기당 평균 종목 {ivol.groupby("qtr").size().mean():.0f}')
    print(ivol['ivol'].describe()[['mean','std','min','50%','max']].round(4).to_dict())
    ivol.to_parquet(IVOL_OUT)

ivol_quarterly.parquet loaded. rows 70332  종목 953
분기당 평균 종목 664
{'mean': 0.0278, 'std': 0.0171, 'min': 0.0, '50%': 0.0235, 'max': 0.4026}


In [25]:
# 24

financials = pd.read_parquet('financials.parquet')
prices = pd.read_parquet('prices.parquet')
ivol = pd.read_parquet('ivol_quarterly.parquet')

fv, ind = compute_fvol(financials, deflator='saleq')
fv, col = finalize_fvol(fv, ind, suffix='')

sig = fv[['code', 'date', 'FVOL']].dropna().copy()
sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1
sig['prev_qtr'] = sig['form_qtr'] - 1
sig = sig.merge(ivol.rename(columns={'qtr': 'prev_qtr'}),
                on=['code', 'prev_qtr'], how='left')

px = prices.copy()
px['ret'] = px['ret'] / 100.0
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

f0 = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
f0 = f0[f0['price'] >= 1000].copy()
f = f0.dropna(subset=['ivol']).copy()

f['fq'] = f.groupby('form_qtr')['FVOL'].transform(
    lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
f['iq'] = f.groupby('form_qtr')['ivol'].transform(
    lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
f = f.dropna(subset=['fq', 'iq'])
f['fq'] = f['fq'].astype(int)
f['iq'] = f['iq'].astype(int)
f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

print(f'가격필터 후 {len(f0)}  ivol 부착 {len(f)} ({len(f)/len(f0)*100:.1f}%)  '
      f'분기당 {f.groupby("form_qtr").size().mean():.0f}종목')

holds = []
for k in range(1, 25):
    h = f[['code', 'form_ym', 'fq', 'iq']].copy()
    h['hold_ym'] = h['form_ym'] + k
    h['qt'] = (k - 1) // 3 + 1
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

ret_m = px[['code', 'ym', 'ret']].dropna()
m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
m = m.drop_duplicates(['code', 'form_ym', 'ym', 'qt'])
cell = m.groupby(['qt', 'ym', 'iq', 'fq'])['ret'].mean().reset_index(name='r')

def tstat(s):
    s = s.dropna()
    return s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))

rA, partsA = [], []
for qt in range(1, 9):
    sub = cell[cell['qt'] == qt]
    lhs = []
    for iqv in range(1, 6):
        w = sub[sub['iq'] == iqv].pivot(index='ym', columns='fq', values='r')
        if 1 in w and 5 in w:
            lhs.append((w[1] - w[5]).rename(iqv))
    lh = pd.concat(lhs, axis=1).mean(axis=1).dropna()
    partsA.append(lh)
    rA.append({'Q': f'Q{qt}', 'L-H%': lh.mean() * 100, 't': tstat(lh), '월수': len(lh)})
jtA = pd.concat(partsA, axis=1).mean(axis=1).dropna()
rA.append({'Q': 'JT8', 'L-H%': jtA.mean() * 100, 't': tstat(jtA), '월수': len(jtA)})

rB, partsB = [], []
for qt in range(1, 9):
    sub = cell[cell['qt'] == qt]
    lhs = []
    for fqv in range(1, 6):
        w = sub[sub['fq'] == fqv].pivot(index='ym', columns='iq', values='r')
        if 1 in w and 5 in w:
            lhs.append((w[1] - w[5]).rename(fqv))
    lh = pd.concat(lhs, axis=1).mean(axis=1).dropna()
    partsB.append(lh)
    rB.append({'Q': f'Q{qt}', 'L-H%': lh.mean() * 100, 't': tstat(lh), '월수': len(lh)})
jtB = pd.concat(partsB, axis=1).mean(axis=1).dropna()
rB.append({'Q': 'JT8', 'L-H%': jtB.mean() * 100, 't': tstat(jtB), '월수': len(jtB)})

grid_q1 = (cell[cell['qt'] == 1].groupby(['iq', 'fq'])['r'].mean() * 100).unstack('fq')
grid_q1.index.name = 'IVOL분위'
grid_q1.columns.name = 'FVOL분위'

print('Panel A: IVOL 통제 후 FVOL Low-High (EW, %/월)')
display(pd.DataFrame(rA).round(2))
print('Panel B: FVOL 통제 후 IVOL Low-High (EW, %/월)')
display(pd.DataFrame(rB).round(2))
print('25구간 평균수익률 (Q1 기준, %/월) — 행=IVOL분위, 열=FVOL분위')
display(grid_q1.round(3))

가격필터 후 61778  ivol 부착 60187 (97.4%)  분기당 614종목
Panel A: IVOL 통제 후 FVOL Low-High (EW, %/월)


,Q,L-H%,t,월수
0,Q1,0.51,3.12,291
1,Q2,0.40,2.35,288
2,Q3,0.25,1.47,285
3,Q4,0.11,0.63,282
4,Q5,0.07,0.38,279
5,Q6,0.12,0.65,276
6,Q7,0.05,0.29,273
7,Q8,-0.08,-0.44,270
8,JT8,0.24,1.54,291


Panel B: FVOL 통제 후 IVOL Low-High (EW, %/월)


,Q,L-H%,t,월수
0,Q1,0.43,1.66,291
1,Q2,0.14,0.56,288
2,Q3,0.62,2.72,285
3,Q4,0.66,2.90,282
4,Q5,0.72,3.62,279
5,Q6,0.31,1.50,276
6,Q7,0.17,0.76,273
7,Q8,0.35,1.81,270
8,JT8,0.47,2.44,291


25구간 평균수익률 (Q1 기준, %/월) — 행=IVOL분위, 열=FVOL분위


FVOL분위,1,2,3,4,5
IVOL분위,,,,,
1,1.142,1.366,1.311,1.416,0.802
2,1.168,1.235,1.280,1.197,1.068
3,1.396,1.172,1.084,1.216,0.934
4,1.609,1.555,1.343,1.399,0.914
5,1.000,0.839,1.029,0.978,0.053


In [26]:
# 25

prices = pd.read_parquet('prices.parquet')
market = pd.read_parquet('market.parquet')

MKT_OUT = 'mkt_factor.parquet'

if Path(MKT_OUT).exists():
    mktf = pd.read_parquet(MKT_OUT)
    print(f'{MKT_OUT} loaded. rows {len(mktf)}  기간 {mktf["ym"].min()} ~ {mktf["ym"].max()}')
    display(mktf[['mkt_ew', 'rf_m', 'mktrf_ew']].describe().loc[['mean', 'std', 'min', 'max']].round(5))
else:
    px = prices[['code', 'date', 'ret']].dropna(subset=['ret']).copy()
    px['ym'] = pd.PeriodIndex(px['date'], freq='M')
    ew = px.groupby('ym')['ret'].mean().div(100).rename('mkt_ew').reset_index()

    mk = market.copy()
    mk['date'] = pd.to_datetime(mk['date'])
    mk['ym'] = pd.PeriodIndex(mk['date'], freq='M')
    rf_m = (mk.groupby('ym')['rf'].last() / 100 / 12).rename('rf_m').reset_index()

    mktf = ew.merge(rf_m, on='ym', how='inner')
    mktf['mktrf_ew'] = mktf['mkt_ew'] - mktf['rf_m']
    mktf = mktf.sort_values('ym').reset_index(drop=True)

    print(f'{MKT_OUT} written. rows {len(mktf)}  기간 {mktf["ym"].min()} ~ {mktf["ym"].max()}')
    display(mktf[['mkt_ew', 'rf_m', 'mktrf_ew']].describe().loc[['mean', 'std', 'min', 'max']].round(5))
    mktf.to_parquet(MKT_OUT)

mkt_factor.parquet loaded. rows 319  기간 1999-12 ~ 2026-06


,mkt_ew,rf_m,mktrf_ew
mean,0.01259,0.00268,0.00990
std,0.06733,0.00129,0.06743
min,-0.30789,0.00052,-0.31287
max,0.26956,0.00612,0.26461


In [27]:
# 26

FALL = pd.read_excel('kospi_data.xlsx', sheet_name='35_Fall')
FTHM = pd.read_excel('kospi_data.xlsx', sheet_name='37_Fthm')

JKP_OUT = 'jkp_factors.parquet'

FALL_FACTORS = ['market_equity', 'be_me', 'ope_be', 'at_gr1', 'ni_be', 'niq_be']
THM_FACTORS = ['size']

if Path(JKP_OUT).exists():
    jkp = pd.read_parquet(JKP_OUT)
    print(f'{JKP_OUT} loaded. shape {jkp.shape}  기간 {jkp["ym"].min()} ~ {jkp["ym"].max()}')
    display(jkp.describe().loc[['mean', 'std']].round(5))
else:
    a = FALL[FALL['name'].isin(FALL_FACTORS)][['name', 'date', 'ret']].copy()
    t = FTHM[FTHM['name'].isin(THM_FACTORS)][['name', 'date', 'ret']].copy()
    t['name'] = t['name'] + '_thm'
    both = pd.concat([a, t], ignore_index=True)
    both['ym'] = pd.PeriodIndex(pd.to_datetime(both['date']), freq='M')

    jkp = both.pivot_table(index='ym', columns='name', values='ret').reset_index()
    jkp.columns.name = None
    jkp = jkp.sort_values('ym').reset_index(drop=True)

    order = ['ym'] + FALL_FACTORS + ['size_thm']
    jkp = jkp[[c for c in order if c in jkp.columns]]

    print(f'{JKP_OUT} written. shape {jkp.shape}  기간 {jkp["ym"].min()} ~ {jkp["ym"].max()}')
    print('팩터별 유효 관측수(비결측):')
    display(jkp.drop(columns='ym').notna().sum().rename('n_obs').to_frame())
    print('팩터별 평균/표준편차(%/월):')
    display((jkp.drop(columns='ym').agg(['mean', 'std']) * 100).round(4))
    jkp.to_parquet(JKP_OUT)

jkp_factors.parquet loaded. shape (450, 8)  기간 1988-07 ~ 2025-12


,market_equity,be_me,ope_be,at_gr1,ni_be,niq_be,size_thm
mean,0.00457,0.01580,0.00444,0.00396,0.00152,0.00215,0.00667
std,0.06494,0.04711,0.02966,0.02730,0.03574,0.03640,0.05565


In [28]:
# 27

financials = pd.read_parquet('financials.parquet')
prices = pd.read_parquet('prices.parquet')

FVOLRET_OUT = 'fvol_factor_ret.parquet'

if Path(FVOLRET_OUT).exists():
    fvret = pd.read_parquet(FVOLRET_OUT)
    print(f'{FVOLRET_OUT} loaded. rows {len(fvret)}  기간 {fvret["ym"].min()} ~ {fvret["ym"].max()}')
    display((fvret[['fvol_lh']].describe().loc[['mean', 'std', 'min', 'max']] * 100).round(4))
else:
    fv, ind = compute_fvol(financials, deflator='saleq')
    fv, col = finalize_fvol(fv, ind, suffix='')

    sig = fv[['code', 'date', 'FVOL']].dropna().copy()
    sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

    px = prices.copy()
    px['ret'] = px['ret'] / 100.0
    px['ym'] = pd.PeriodIndex(px['date'], freq='M')
    form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
    form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

    f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
    f = f[f['price'] >= 1000].copy()
    f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)
    f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')
    f = f[f['decile'].isin([1, 10])][['code', 'form_ym', 'decile']]

    holds = []
    for k in range(1, 4):
        h = f.copy()
        h['hold_ym'] = h['form_ym'] + k
        holds.append(h)
    hold = pd.concat(holds, ignore_index=True)

    ret_m = px[['code', 'ym', 'ret']].dropna()
    m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
    m = m.drop_duplicates(['code', 'form_ym', 'ym'])
    port = m.groupby(['ym', 'decile'])['ret'].mean().unstack('decile')

    fvret = pd.DataFrame({'ym': port.index})
    fvret['low_d1'] = port[1].values
    fvret['high_d10'] = port[10].values
    fvret['fvol_lh'] = fvret['low_d1'] - fvret['high_d10']
    fvret = fvret.dropna(subset=['fvol_lh']).sort_values('ym').reset_index(drop=True)

    print(f'{FVOLRET_OUT} written. rows {len(fvret)}  기간 {fvret["ym"].min()} ~ {fvret["ym"].max()}')
    print(f'fvol_lh 평균 {fvret["fvol_lh"].mean()*100:.4f}%/월  t≈{fvret["fvol_lh"].mean()/(fvret["fvol_lh"].std()/len(fvret)**0.5):.2f}')
    display((fvret[['low_d1', 'high_d10', 'fvol_lh']].describe().loc[['mean', 'std', 'min', 'max']] * 100).round(4))
    fvret.to_parquet(FVOLRET_OUT)

fvol_factor_ret.parquet loaded. rows 291  기간 2002-04 ~ 2026-06


,fvol_lh
mean,0.7897
std,4.8071
min,-17.6979
max,15.8022


In [29]:
# 28

mktf = pd.read_parquet('mkt_factor.parquet')
jkp = pd.read_parquet('jkp_factors.parquet')
fvret = pd.read_parquet('fvol_factor_ret.parquet')

df = fvret[['ym', 'fvol_lh']].merge(mktf[['ym', 'mktrf_ew']], on='ym', how='inner')
df = df.merge(jkp, on='ym', how='inner')

MODELS = {
    'FF5 main':   ['mktrf_ew', 'market_equity', 'be_me', 'ope_be', 'at_gr1'],
    'FF5 robust': ['mktrf_ew', 'size_thm',      'be_me', 'ope_be', 'at_gr1'],
    'q main':     ['mktrf_ew', 'market_equity', 'at_gr1', 'ni_be'],
    'q robust':   ['mktrf_ew', 'market_equity', 'at_gr1', 'niq_be'],
}

def run_ols(y, X):
    X1 = np.column_stack([np.ones(len(X)), X])
    beta, _, _, _ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ beta
    n, k = X1.shape
    sigma2 = (resid @ resid) / (n - k)
    XtX_inv = np.linalg.inv(X1.T @ X1)
    se = np.sqrt(np.diag(sigma2 * XtX_inv))
    tstat = beta / se
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - (resid @ resid) / ss_tot
    return beta, tstat, r2, n

rows_alpha = {}
betas_store = {}
for name, facs in MODELS.items():
    sub = df[['fvol_lh'] + facs].dropna()
    y = sub['fvol_lh'].values
    X = sub[facs].values
    beta, tstat, r2, n = run_ols(y, X)
    rows_alpha[name] = {
        'alpha_%m': round(beta[0] * 100, 4),
        't_alpha': round(tstat[0], 2),
        'R2': round(r2, 4),
        'n_month': n,
    }
    betas_store[name] = (facs, beta, tstat)

tbl_alpha = pd.DataFrame(rows_alpha).T
print('표1. Spanning α 요약 (FVOL Low-High, EW, OLS)')
display(tbl_alpha)

ALL_FACS = ['mktrf_ew', 'market_equity', 'size_thm', 'be_me', 'ope_be', 'at_gr1', 'ni_be', 'niq_be']

rows_beta = {}
for name, (facs, beta, tstat) in betas_store.items():
    d = {'const': round(beta[0] * 100, 4)}
    for i, f in enumerate(facs):
        d[f] = round(beta[i + 1], 3)
        d[f + '_t'] = round(tstat[i + 1], 2)
    rows_beta[name] = d

tbl_beta = pd.DataFrame(rows_beta).T
col_order = ['const']
for f in ALL_FACS:
    if f in tbl_beta.columns:
        col_order += [f, f + '_t']
tbl_beta = tbl_beta[[c for c in col_order if c in tbl_beta.columns]]

print('표2. 팩터 베타 상세 (계수 및 t, 빈칸=해당 모형 미사용)')
display(tbl_beta)

표1. Spanning α 요약 (FVOL Low-High, EW, OLS)


,alpha_%m,t_alpha,R2,n_month
FF5 main,0.4169,1.57,0.3019,285.0
FF5 robust,0.4396,1.67,0.3140,285.0
q main,0.9464,3.52,0.2016,285.0
q robust,0.7194,2.86,0.2378,217.0


표2. 팩터 베타 상세 (계수 및 t, 빈칸=해당 모형 미사용)


,const,mktrf_ew,mktrf_ew_t,market_equity,market_equity_t,size_thm,size_thm_t,be_me,be_me_t,ope_be,ope_be_t,at_gr1,at_gr1_t,ni_be,ni_be_t,niq_be,niq_be_t
FF5 main,0.4169,-0.257,-6.09,-0.155,-2.26,NaN,NaN,0.439,5.86,0.381,2.97,-0.361,-2.44,NaN,NaN,NaN,NaN
FF5 robust,0.4396,-0.255,-6.19,NaN,NaN,-0.275,-3.18,0.489,6.31,0.337,2.66,-0.350,-2.39,NaN,NaN,NaN,NaN
q main,0.9464,-0.217,-4.75,-0.097,-1.37,NaN,NaN,NaN,NaN,NaN,NaN,-0.037,-0.24,0.454,3.1,NaN,NaN
q robust,0.7194,-0.275,-6.14,-0.123,-1.79,NaN,NaN,NaN,NaN,NaN,NaN,0.087,0.58,NaN,NaN,0.246,3.17


In [30]:
# 29

mktf = pd.read_parquet('mkt_factor.parquet')
jkp = pd.read_parquet('jkp_factors.parquet')
fvret = pd.read_parquet('fvol_factor_ret.parquet')

df = fvret[['ym', 'fvol_lh']].merge(mktf[['ym', 'mktrf_ew']], on='ym', how='inner')
df = df.merge(jkp, on='ym', how='inner')

MODELS = {
    'q (기본)':        ['mktrf_ew', 'market_equity', 'at_gr1', 'ni_be'],
    'q + 가치(be_me)': ['mktrf_ew', 'market_equity', 'at_gr1', 'ni_be', 'be_me'],
    'FF5 (기본)':      ['mktrf_ew', 'market_equity', 'be_me', 'ope_be', 'at_gr1'],
}

def run_ols(y, X):
    X1 = np.column_stack([np.ones(len(X)), X])
    beta, _, _, _ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ beta
    n, k = X1.shape
    sigma2 = (resid @ resid) / (n - k)
    se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X1.T @ X1)))
    return beta, beta / se, n

rows_addval = {}
for name, facs in MODELS.items():
    sub = df[['fvol_lh'] + facs].dropna()
    beta, tstat, n = run_ols(sub['fvol_lh'].values, sub[facs].values)
    rows_addval[name] = {
        'alpha_%m': round(beta[0] * 100, 4),
        't_alpha': round(tstat[0], 2),
        'be_me_계수': round(beta[facs.index('be_me') + 1], 3) if 'be_me' in facs else np.nan,
        'be_me_t': round(tstat[facs.index('be_me') + 1], 2) if 'be_me' in facs else np.nan,
        'n_month': n,
    }

tbl_addval = pd.DataFrame(rows_addval).T
print('q팩터에 가치(be_me) 추가 시 α 변화')
display(tbl_addval)

q팩터에 가치(be_me) 추가 시 α 변화


,alpha_%m,t_alpha,be_me_계수,be_me_t,n_month
q (기본),0.9464,3.52,NaN,NaN,285.0
q + 가치(be_me),0.4237,1.60,0.464,6.26,285.0
FF5 (기본),0.4169,1.57,0.439,5.86,285.0


In [31]:
# 30

TESTASSET_OUT = 'test_assets_ext.parquet'

CHARS = [
    'be_me','at_me','ni_me','sale_me','fcf_me','div12m_me',
    'ope_be','ni_be','gp_at','cop_at','ebit_sale','op_at',
    'at_gr1','be_gr1a','capx_gr1','inv_gr1','noa_at','ppeinv_gr1a',
    'sale_gr1','lnoa_gr1a','col_gr1a',
    'ret_12_1','ret_6_1','ret_1_0','ret_60_12','seas_1_1na','resff3_12_1',
    'ivol_capm_252d','rvol_252d','beta_60m','rmax5_21d','rskew_21d',
    'ami_126d','dolvol_126d','zero_trades_252d','turnover_126d',
    'oaccruals_at','taccruals_at','qmj',
]
N_SIZE = 5
N_CHAR = 5
MIN_BUCKET = 5
MIN_MONTHS = 60

if Path(TESTASSET_OUT).exists():
    ta = pd.read_parquet(TESTASSET_OUT)
    print(f'{TESTASSET_OUT} loaded. shape {ta.shape}  자산 {ta.shape[1]-1}개  기간 {ta["ym"].min()} ~ {ta["ym"].max()}')
else:
    jkp = pd.read_parquet('jkp_kor_chars_ext_c.parquet')

    d = jkp[['gvkey', 'iid', 'eom', 'ret_exc', 'market_equity'] + CHARS].copy()
    d['ym'] = pd.PeriodIndex(d['eom'], freq='M')
    d['hold_ym'] = d['ym'] + 1

    def winz(s):
        lo, hi = s.quantile(0.01), s.quantile(0.99)
        return s.clip(lo, hi)
    for c in CHARS + ['market_equity']:
        d[c] = d.groupby('ym')[c].transform(winz)

    nxt = d[['gvkey', 'iid', 'ym', 'ret_exc']].rename(
        columns={'ym': 'hold_ym', 'ret_exc': 'r_next'})
    d = d.merge(nxt, on=['gvkey', 'iid', 'hold_ym'], how='inner')
    d = d.dropna(subset=['r_next', 'market_equity'])

    def qbin(x, n):
        return pd.qcut(x, n, labels=False, duplicates='drop')

    d['sq'] = d.groupby('ym')['market_equity'].transform(lambda x: qbin(x, N_SIZE)) + 1

    port = {}
    for c in CHARS:
        sub = d.dropna(subset=[c]).copy()
        sub['cq'] = (sub.groupby(['ym', 'sq'])[c]
                     .transform(lambda x: qbin(x, N_CHAR)) + 1)
        sub = sub.dropna(subset=['cq'])
        sub['sq'] = sub['sq'].astype(int)
        sub['cq'] = sub['cq'].astype(int)

        grp = sub.groupby(['ym', 'sq', 'cq'])
        ew = grp['r_next'].mean()
        cnt = grp['r_next'].size()
        ew = ew[cnt >= MIN_BUCKET]

        for sq in range(1, N_SIZE + 1):
            for cq in range(1, N_CHAR + 1):
                try:
                    port[f'{c}__s{sq}c{cq}'] = ew.xs((sq, cq), level=('sq', 'cq'))
                except KeyError:
                    pass

    ta = pd.DataFrame(port).sort_index()
    ta.index.name = 'ym'
    valid_cols = [c for c in ta.columns if ta[c].notna().sum() >= MIN_MONTHS]
    ta = ta[valid_cols].reset_index()

    print(f'{TESTASSET_OUT} written. shape {ta.shape}  자산 {ta.shape[1]-1}개  '
          f'기간 {ta["ym"].min()} ~ {ta["ym"].max()}')

    ta.to_parquet(TESTASSET_OUT)

test_assets_ext.parquet loaded. shape (299, 976)  자산 975개  기간 2001-01 ~ 2025-11


In [32]:
# 31

ZOO_OUT = 'zoo_factors.parquet'

if Path(ZOO_OUT).exists():
    zoo = pd.read_parquet(ZOO_OUT)
    print(f'{ZOO_OUT} loaded. shape {zoo.shape}  팩터 {zoo.shape[1]-1}개  기간 {zoo["ym"].min()} ~ {zoo["ym"].max()}')
else:
    fall = pd.read_excel('kospi_data.xlsx', sheet_name='35_Fall')
    fall = fall[fall['location'] == 'kor'][['name', 'date', 'ret']].copy()
    fall['ym'] = pd.PeriodIndex(pd.to_datetime(fall['date']), freq='M')

    zoo = fall.pivot_table(index='ym', columns='name', values='ret')
    zoo.columns.name = None

    mktf = pd.read_parquet('mkt_factor.parquet')
    zoo = zoo.merge(mktf[['ym', 'mktrf_ew']], left_index=True, right_on='ym', how='left').set_index('ym')
    zoo = zoo.sort_index().reset_index()

    print(f'{ZOO_OUT} written. shape {zoo.shape}  팩터 {zoo.shape[1]-1}개  '
          f'기간 {zoo["ym"].min()} ~ {zoo["ym"].max()}')

    zoo.to_parquet(ZOO_OUT)

zoo_factors.parquet loaded. shape (450, 155)  팩터 154개  기간 1988-07 ~ 2025-12


In [34]:
# 32

MIN_FAC_MONTHS = 284
N_SEED = 200
N_ALPHA = 60
N_JOBS = -1

ta = pd.read_parquet('test_assets_ext.parquet')
zoo = pd.read_parquet('zoo_factors.parquet')
fvret = pd.read_parquet('fvol_factor_ret.parquet')[['ym', 'fvol_lh']]

R_all = ta.drop(columns='ym').to_numpy(dtype=float)
mask = ~np.isnan(R_all).any(axis=1)
Rdim = R_all[mask]
Td, nd = Rdim.shape
Rc = Rdim - Rdim.mean(0)
eig = np.linalg.eigvalsh((Rc.T @ Rc) / (Td - 1))[::-1]
cum = np.cumsum(eig) / eig.sum()

dim_tbl = pd.DataFrame({
    '주성분 축': [1, 2, 3, 5, 10],
    '누적 설명력(%)': [round(cum[k-1]*100, 1) for k in [1, 2, 3, 5, 10]],
}).set_index('주성분 축')

fac_all = [c for c in zoo.columns if c != 'ym']
common_ym = sorted(set(ta['ym']) & set(fvret['ym']))
nobs = zoo[zoo['ym'].isin(common_ym)][fac_all].notna().sum()
fac_cols = [c for c in fac_all if nobs[c] >= MIN_FAC_MONTHS]

Z = zoo[zoo['ym'].isin(common_ym)][['ym'] + fac_cols].dropna()
A = ta[ta['ym'].isin(common_ym)]
F = fvret[fvret['ym'].isin(common_ym)]
panel = Z.merge(F, on='ym', how='inner').merge(A, on='ym', how='inner').dropna()
asset_cols = [c for c in A.columns if c != 'ym']

H = panel[fac_cols].to_numpy(dtype=float)
R = panel[asset_cols].to_numpy(dtype=float)
T, n, p = len(panel), R.shape[1], H.shape[1]

rbar = R.mean(0)
Rd = R - R.mean(0); Hd = H - H.mean(0)
vh = (Hd ** 2).sum(0) / (T - 1)
Bh = (Rd.T @ Hd) / (T - 1) / vh
nrm = np.sqrt((Bh ** 2).sum(0)); nrm[nrm == 0] = 1.0
Xf = Bh / nrm

alphas, coefs_full, _ = lasso_path(Xf, rbar, n_alphas=N_ALPHA)
nsel_path = (np.abs(coefs_full) > 1e-10).sum(0)

def seed_err(seed):
    err = np.zeros(len(alphas))
    rng = np.random.RandomState(seed)
    folds = np.array_split(rng.permutation(T), 10)
    for k in range(10):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(10) if j != k])
        rt, Ht = R[tr], H[tr]; rv, Hv = R[te], H[te]
        Rdt = rt - rt.mean(0); Hdt = Ht - Ht.mean(0)
        Bht = (Rdt.T @ Hdt) / (len(tr) - 1) / ((Hdt ** 2).sum(0) / (len(tr) - 1)) / nrm
        Rdv = rv - rv.mean(0); Hdv = Hv - Hv.mean(0)
        Bhv = (Rdv.T @ Hdv) / (len(te) - 1) / ((Hdv ** 2).sum(0) / (len(te) - 1)) / nrm
        _, cf, _ = lasso_path(Bht, rt.mean(0), alphas=alphas, max_iter=5000)
        pred = Bhv @ cf + (rt.mean(0).mean() - Bht.mean(0) @ cf)
        err += ((rv.mean(0)[:, None] - pred) ** 2).mean(0)
    return err, alphas[err.argmin()]

out = Parallel(n_jobs=N_JOBS)(delayed(seed_err)(s) for s in range(N_SEED))
err = sum(o[0] for o in out) / N_SEED
best_alphas = [o[1] for o in out]
seed_nsel = [int(nsel_path[np.where(alphas == a)[0][0]]) for a in best_alphas]

idx = range(0, len(alphas), 5)
cv_tbl = pd.DataFrame({
    '선택 팩터 수': [int(nsel_path[i]) for i in idx],
    '평균 CV error': [round(err[i], 6) for i in idx],
})

seed_tbl = (pd.Series(seed_nsel).value_counts().sort_index()
            .rename_axis('선택 팩터 수').reset_index(name='시드 수'))

summary = pd.DataFrame(
    {'값': [nd, round(cum[0]*100, 1), p, T, int(nsel_path[err.argmin()]), f'{seed_nsel.count(0)}/{N_SEED}']},
    index=['테스트 자산 수', '제1주성분 설명력(%)', '후보 팩터 수', '공통 표본월',
           'CV 선택 팩터 수(전체표본)', '팩터 미선택 시드'],
)

print('▎테스트 자산 주성분 구조')
display(dim_tbl)
print('▎FGX 1단계 CV: 페널티별 선택 팩터 수와 CV error')
display(cv_tbl)
print('▎200개 시드별 선택 팩터 수 분포')
display(seed_tbl)
print('▎요약')
display(summary)

▎테스트 자산 주성분 구조


,누적 설명력(%)
주성분 축,
1,85.1
2,89.1
3,90.3
5,91.6
10,93.1


▎FGX 1단계 CV: 페널티별 선택 팩터 수와 CV error


,선택 팩터 수,평균 CV error
0,0,0.006748
1,3,0.012268
2,6,0.015765
3,8,0.016418
4,8,0.014962
5,11,0.012807
6,15,0.013744
7,27,0.017076
8,38,0.019788
9,46,0.020877


▎200개 시드별 선택 팩터 수 분포


,선택 팩터 수,시드 수
0,0,191
1,12,2
2,13,2
3,15,1
4,19,1
5,45,1
6,68,1
7,76,1


▎요약


,값
테스트 자산 수,975
제1주성분 설명력(%),85.1
후보 팩터 수,131
공통 표본월,223
CV 선택 팩터 수(전체표본),0
팩터 미선택 시드,191/200


In [ ]:
# # 시각화

# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False
# NAVY = '#1F3864'

# fv, ind = compute_fvol(financials, deflator='saleq')
# fv, col = finalize_fvol(fv, ind, suffix='')

# d = fv.sort_values(['code','date']).copy()
# g = d.groupby('code', sort=False)
# d['atq_lag1'] = g['atq'].shift(1)
# d['qbe'] = d['seqq'].fillna(0) + d['txditcq'].fillna(0) - d['pstkq'].fillna(0)
# d['qbe_lag1'] = g['qbe'].shift(1)
# d['GP'] = (d['saleq'] - d['cogsq']) / d['atq_lag1']
# d['OP'] = (d['saleq'] - d['cogsq'] - d['xsgaq'] - d['xintq']) / d['qbe_lag1']
# d['ROE'] = d['ibq'] / d['qbe_lag1']
# for c in ['GP','OP','ROE']:
#     d[c] = d[c].replace([np.inf,-np.inf], np.nan)

# fin_has = financials.groupby('code')['atq'].apply(lambda s: s.notna().any())
# fin_codes = set(fin_has[fin_has].index)
# dfin = delisted[delisted['code'].isin(fin_codes)].copy()
# dist = dfin['del_date'].dt.year.value_counts().sort_index()
# fig, ax = plt.subplots(figsize=(9, 4.2))
# ax.bar(dist.index, dist.values, color=NAVY, width=0.7)
# ax.set_xlabel('상장폐지 연도'); ax.set_ylabel('기업 수')
# ax.set_title('그림 1: 상장폐지 기업의 연도별 분포 (재무 데이터 존재 235개)', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# gq = fv[fv['FVOL'].notna()].groupby('date').size()
# fig, ax = plt.subplots(figsize=(9, 4.2))
# ax.plot(gq.index, gq.values, color=NAVY, linewidth=2)
# ax.fill_between(gq.index, gq.values, alpha=0.15, color=NAVY)
# ax.axhline(500, color='#C00000', linestyle='--', linewidth=1, label='500개 기준선')
# ax.set_xlabel('연도'); ax.set_ylabel('유효 기업 수')
# ax.set_title('그림 2: 분기별 유효 기업 수 추이', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# ALL_ITEMS = ['actq','rectq','invtq','ppentq','atq','lctq','dlttq','dlcq','ltq',
#              'seqq','cogsq','xsgaq','saleq','apq']
# miss = {c: financials[c].isna().mean()*100 for c in ALL_ITEMS}
# miss = pd.Series(miss).sort_values()
# colors = ['#C00000' if c=='dlttq' else NAVY for c in miss.index]
# fig, ax = plt.subplots(figsize=(9, 4.5))
# ax.barh(range(len(miss)), miss.values, color=colors)
# ax.set_yticks(range(len(miss))); ax.set_yticklabels(miss.index)
# ax.set_xlabel('결측 비율 (%)')
# ax.set_title('그림 3: 재무 항목별 결측 수준', fontweight='bold', color=NAVY)
# ax.grid(axis='x', alpha=0.3)
# plt.tight_layout(); plt.show()

# v = d[d['FVOL'].notna()].copy()
# v['dec'] = v.groupby('date')['FVOL'].transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop'))
# series = {c: v.groupby(['date','dec'])[c].median().reset_index().groupby('dec')[c].mean() for c in ['GP','OP','ROE']}
# fig, ax = plt.subplots(figsize=(9, 4.8))
# labels = {'GP':'총이익성 (GP)', 'OP':'영업수익성 (OP)', 'ROE':'자기자본이익률 (ROE)'}
# colors = {'GP':NAVY, 'OP':'#C55A11', 'ROE':'#548235'}
# markers = {'GP':'o', 'OP':'s', 'ROE':'^'}
# for c in ['GP','OP','ROE']:
#     ax.plot(range(10), series[c].values, marker=markers[c], color=colors[c], label=labels[c], linewidth=2, markersize=6)
# ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
# ax.set_xticks(range(10)); ax.set_xticklabels(['0\n(최저)','1','2','3','4','5','6','7','8','9\n(최고)'])
# ax.set_xlabel('FVOL 분위'); ax.set_ylabel('수익성')
# ax.set_title('그림 4: FVOL 분위별 수익성 지표', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# fv2, ind2 = compute_fvol(financials, deflator='atq')
# fv2, _ = finalize_fvol(fv2, ind2, suffix='')
# d2 = d.merge(fv2[['code','date','FVOL']].rename(columns={'FVOL':'FVOL_a'}), on=['code','date'])
# v2 = d2[d2['FVOL_a'].notna()].copy()
# v2['dec'] = v2.groupby('date')['FVOL_a'].transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop'))
# gp_sale = series['GP']
# gp_asset = v2.groupby(['date','dec'])['GP'].median().reset_index().groupby('dec')['GP'].mean()
# fig, ax = plt.subplots(figsize=(9, 4.5))
# ax.plot(range(10), gp_sale.values, marker='o', color=NAVY, linewidth=2, label='매출 기준')
# ax.plot(range(10), gp_asset.values, marker='s', color='#B0B0B0', linewidth=2, label='총자산 기준')
# ax.set_xticks(range(10)); ax.set_xticklabels(['0\n(최저)','1','2','3','4','5','6','7','8','9\n(최고)'])
# ax.set_xlabel('FVOL 분위'); ax.set_ylabel('총이익성 (GP)')
# ax.set_title('그림 5: 규모 보정 기준별 수익성 구분력', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# dd = fv[['code','date','FVOL']].dropna().copy()
# pq = pd.PeriodIndex(dd['date'], freq='Q')
# dd['qidx'] = pq.year * 4 + (pq.quarter - 1)
# lags = [1,2,4,8,12,20]; corrs = []
# for lag in lags:
#     a = dd.copy(); a['fut'] = a['qidx'] + lag
#     m = a.merge(a[['code','qidx','FVOL']].rename(columns={'qidx':'fut','FVOL':'FVOL_f'}), on=['code','fut'], how='inner')
#     corrs.append(m[['FVOL','FVOL_f']].corr().iloc[0,1])
# fig, ax = plt.subplots(figsize=(9, 4.2))
# years = [l/4 for l in lags]
# ax.plot(years, corrs, marker='o', color=NAVY, linewidth=2, markersize=7)
# for x,y in zip(years, corrs):
#     ax.annotate(f'{y:.2f}', (x,y), textcoords='offset points', xytext=(0,10), ha='center', fontsize=9)
# ax.set_xlabel('경과 기간 (년)'); ax.set_ylabel('FVOL 자기상관'); ax.set_ylim(0, 1)
# ax.set_title('그림 6: FVOL의 지속성', fontweight='bold', color=NAVY)
# ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# rcols = [f'rank_{c}' for c in FVOL_COLS]
# valid = fv[fv['FVOL'].notna()]
# contrib = {c: valid[[f'rank_{c}','FVOL']].corr().iloc[0,1] for c in FVOL_COLS}
# contrib = pd.Series(contrib).sort_values()
# colors = ['#C00000' if c=='dlttq' else NAVY for c in contrib.index]
# fig, ax = plt.subplots(figsize=(9, 4.8))
# ax.barh(range(len(contrib)), contrib.values, color=colors)
# ax.set_yticks(range(len(contrib))); ax.set_yticklabels(contrib.index)
# ax.set_xlabel('종합 지표와의 연관성')
# ax.set_title('그림 7: 재무 항목별 지표 기여도', fontweight='bold', color=NAVY)
# ax.grid(axis='x', alpha=0.3)
# plt.tight_layout(); plt.show()

# rank_corr = valid[rcols].corr()
# labels = list(FVOL_COLS)
# fig, ax = plt.subplots(figsize=(8.5, 7))
# im = ax.imshow(rank_corr.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
# ax.set_xticks(range(14)); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
# ax.set_yticks(range(14)); ax.set_yticklabels(labels, fontsize=9)
# for i in range(14):
#     for j in range(14):
#         val = rank_corr.values[i,j]
#         ax.text(j, i, f'{val:.2f}', ha='center', va='center', color='white' if val > 0.55 else 'black', fontsize=7)
# ax.set_title('그림 8: 14개 재무 항목 변동성 간 연관성', fontweight='bold', color=NAVY, pad=12)
# fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
# plt.tight_layout(); plt.show()


# # 시각화

# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False
# NAVY = '#1F3864'
# RED = '#C00000'
# GRAY = '#B0B0B0'

# prices = pd.read_parquet('prices.parquet')

# fv, ind = compute_fvol(financials, deflator='saleq')
# fv, col = finalize_fvol(fv, ind, suffix='')

# px = prices.copy()
# px['ret'] = px['ret'] / 100.0
# px['ym'] = pd.PeriodIndex(px['date'], freq='M')
# ret_m = px[['code', 'ym', 'ret']].dropna()

# sig = fv[['code', 'date', 'FVOL']].dropna().copy()
# sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1
# form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price', 'mktcap']].copy()
# form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

# f = sig.merge(form_px[['code', 'form_qtr', 'price', 'mktcap']], on=['code', 'form_qtr'], how='inner')
# f = f[f['price'] >= 1000].copy()
# f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)
# f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

# del_bad = delisted[delisted['reason_cat'].isin(['감사의견', '정리절차', '자본잠식'])]['code']
# pd_del = px[px['code'].isin(del_bad) & px['ret'].notna()].sort_values(['code', 'date'])
# cum12 = pd_del.groupby('code').tail(12).groupby('code')['ret'].apply(lambda s: (1 + s).prod() - 1) * 100
# fig, ax = plt.subplots(figsize=(9, 4.2))
# ax.hist(cum12.clip(-100, 150), bins=40, color=NAVY, edgecolor='white')
# ax.axvline(cum12.median(), color=RED, linestyle='--', linewidth=1.2, label=f'중앙값 {cum12.median():.1f}%')
# ax.set_xlabel('마지막 12개월 누적수익률 (%)'); ax.set_ylabel('기업 수')
# ax.set_title('그림 9: 부실 상장폐지 종목의 마지막 12개월 누적수익률 분포', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# holds = []
# for k in [1, 2, 3]:
#     h = f[['code', 'form_ym', 'decile']].copy()
#     h['hold_ym'] = h['form_ym'] + k
#     holds.append(h)
# hold = pd.concat(holds, ignore_index=True)
# m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
# m = m.drop_duplicates(['code', 'form_ym', 'ym'])
# ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='r')
# wide = ew.pivot(index='ym', columns='decile', values='r')
# dec_mean = wide[list(range(1, 11))].mean() * 100
# fig, ax = plt.subplots(figsize=(9, 4.5))
# colors = [NAVY] * 9 + [RED]
# ax.bar(range(1, 11), dec_mean.values, color=colors, width=0.7)
# ax.set_xticks(range(1, 11)); ax.set_xticklabels(['1\n(최저)'] + [str(i) for i in range(2, 10)] + ['10\n(최고)'])
# ax.set_xlabel('FVOL 분위'); ax.set_ylabel('평균 월수익률 (%)')
# ax.set_title('그림 10: FVOL 분위별 평균 월수익률 (동일가중)', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# lh_m = (wide[1] - wide[10]).dropna()
# cum = (1 + lh_m).cumprod()
# fig, ax = plt.subplots(figsize=(9, 4.5))
# ax.plot(cum.index.to_timestamp(), cum.values, color=NAVY, linewidth=2)
# ax.axhline(1, color='grey', linewidth=0.8, linestyle='--')
# ax.set_xlabel('연도'); ax.set_ylabel('누적수익률 (배)')
# ax.set_title('그림 11: 최저-최고 분위 롱숏 누적수익률 (2002~2026)', fontweight='bold', color=NAVY)
# ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# holds = []
# for k in range(1, 25):
#     h = f[['code', 'form_ym', 'decile']].copy()
#     h['hold_ym'] = h['form_ym'] + k
#     h['qtr'] = (k - 1) // 3 + 1
#     holds.append(h)
# hold = pd.concat(holds, ignore_index=True)
# m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
# m = m.drop_duplicates(['code', 'form_ym', 'ym', 'qtr'])
# ewq = m.groupby(['qtr', 'ym', 'decile'])['ret'].mean().reset_index(name='r')
# lh_q, t_q, lo_q, hi_q = [], [], [], []
# for qtr in range(1, 9):
#     w = ewq[ewq['qtr'] == qtr].pivot(index='ym', columns='decile', values='r')
#     lh = (w[1] - w[10]).dropna()
#     lh_q.append(lh.mean() * 100)
#     t_q.append(lh.mean() / (lh.std(ddof=1) / np.sqrt(len(lh))))
#     lo_q.append(w[1].mean() * 100)
#     hi_q.append(w[10].mean() * 100)

# fig, ax = plt.subplots(figsize=(9, 4.2))
# cols4 = [NAVY if t >= 1.96 else GRAY for t in t_q]
# ax.bar(range(1, 9), lh_q, color=cols4, width=0.6)
# for x, (y, t) in enumerate(zip(lh_q, t_q), start=1):
#     ax.annotate(f't={t:.2f}', (x, y), textcoords='offset points', xytext=(0, 5), ha='center', fontsize=8)
# ax.set_xlabel('보유 분기'); ax.set_ylabel('최저-최고 분위 격차 (%/월)')
# ax.set_title('그림 12: 보유 분기별 수익률 격차 추이 (회색: 비유의)', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# fig, ax = plt.subplots(figsize=(9, 4.2))
# ax.plot(range(1, 9), lo_q, marker='o', color=NAVY, linewidth=2, label='최저 FVOL 분위 (매수)')
# ax.plot(range(1, 9), hi_q, marker='s', color=RED, linewidth=2, label='최고 FVOL 분위 (매도)')
# ax.set_xlabel('보유 분기'); ax.set_ylabel('평균 월수익률 (%)')
# ax.set_title('그림 13: 보유 분기별 최저·최고 분위 수익률', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# fm = f.dropna(subset=['mktcap']).copy()
# qsum = fm.groupby('form_qtr')['mktcap'].transform('sum')
# fm['share'] = fm['mktcap'] / qsum
# share = fm.groupby(['form_qtr', 'decile'])['share'].sum().reset_index()
# avg_share = share.groupby('decile')['share'].mean() * 100
# fig, ax = plt.subplots(figsize=(9, 4.2))
# ax.bar(range(1, 11), avg_share.values, color=NAVY, width=0.7)
# ax.set_xticks(range(1, 11)); ax.set_xticklabels(['1\n(최저)'] + [str(i) for i in range(2, 10)] + ['10\n(최고)'])
# ax.set_xlabel('FVOL 분위'); ax.set_ylabel('시가총액 점유율 (%)')
# ax.set_title('그림 14: FVOL 분위별 시가총액 점유율', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# rp = px[['code', 'ym', 'ret', 'mktcap']].sort_values(['code', 'ym']).copy()
# rp['w'] = rp.groupby('code')['mktcap'].shift(1)
# holds = []
# for k in [1, 2, 3]:
#     h = f[['code', 'form_ym', 'decile']].copy()
#     h['hold_ym'] = h['form_ym'] + k
#     holds.append(h)
# hold = pd.concat(holds, ignore_index=True)
# mv = rp.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
# mv = mv.dropna(subset=['ret', 'w']).drop_duplicates(['code', 'form_ym', 'ym'])
# vw = (mv.groupby(['ym', 'decile'])
#       .apply(lambda g: (g['ret'] * g['w']).sum() / g['w'].sum(), include_groups=False)
#       .reset_index(name='r'))
# wv = vw.pivot(index='ym', columns='decile', values='r')
# lh_vw = (wv[1] - wv[10]).dropna()
# ew_val, ew_t = lh_m.mean() * 100, lh_m.mean() / (lh_m.std(ddof=1) / np.sqrt(len(lh_m)))
# vw_val, vw_t = lh_vw.mean() * 100, lh_vw.mean() / (lh_vw.std(ddof=1) / np.sqrt(len(lh_vw)))
# fig, ax = plt.subplots(figsize=(7, 4.2))
# ax.bar([0, 1], [ew_val, vw_val], color=[NAVY, GRAY], width=0.5)
# for x, (y, t) in enumerate(zip([ew_val, vw_val], [ew_t, vw_t])):
#     ax.annotate(f'{y:.2f}%\n(t={t:.2f})', (x, y), textcoords='offset points', xytext=(0, 5), ha='center', fontsize=9)
# ax.set_xticks([0, 1]); ax.set_xticklabels(['동일가중', '시가총액가중'])
# ax.set_ylabel('최저-최고 분위 격차 (%/월)')
# ax.set_title('그림 15: 가중 방식별 성과 비교', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# fs = f.dropna(subset=['mktcap']).copy()
# fs['size_q'] = fs.groupby('form_qtr')['mktcap'].transform(
#     lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
# fs['fq'] = fs.groupby(['form_qtr', 'size_q'])['FVOL'].transform(
#     lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
# fs = fs.dropna(subset=['fq'])
# holds = []
# for k in [1, 2, 3]:
#     h = fs[['code', 'form_ym', 'size_q', 'fq']].copy()
#     h['hold_ym'] = h['form_ym'] + k
#     holds.append(h)
# hold = pd.concat(holds, ignore_index=True)
# ms = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
# ms = ms.drop_duplicates(['code', 'form_ym', 'ym'])
# cell = ms.groupby(['size_q', 'ym', 'fq'])['ret'].mean().reset_index(name='r')
# sz_lh, sz_t = [], []
# for sq in range(1, 6):
#     w = cell[cell['size_q'] == sq].pivot(index='ym', columns='fq', values='r')
#     lh = (w[1] - w[5]).dropna()
#     sz_lh.append(lh.mean() * 100)
#     sz_t.append(lh.mean() / (lh.std(ddof=1) / np.sqrt(len(lh))))
# fig, ax = plt.subplots(figsize=(9, 4.2))
# cols8 = [NAVY if t >= 1.96 else GRAY for t in sz_t]
# ax.bar(range(1, 6), sz_lh, color=cols8, width=0.55)
# for x, (y, t) in enumerate(zip(sz_lh, sz_t), start=1):
#     ax.annotate(f't={t:.2f}', (x, y), textcoords='offset points', xytext=(0, 5), ha='center', fontsize=9)
# ax.set_xticks(range(1, 6))
# ax.set_xticklabels(['1\n(최소형)', '2', '3', '4', '5\n(최대형)'])
# ax.set_xlabel('시가총액 분위'); ax.set_ylabel('저FVOL-고FVOL 격차 (%/월)')
# ax.set_title('그림 16: 시가총액 분위별 신호 강도 (회색: 비유의)', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# res = pd.read_parquet('combo_full_all.parquet')
# gb = res.groupby('k')['t'].agg(['mean', 'min', 'max'])
# fig, ax = plt.subplots(figsize=(9, 4.5))
# ks = gb.index.values
# ax.fill_between(ks, gb['min'], gb['max'], color=NAVY, alpha=0.15, label='최소~최대 범위')
# ax.plot(ks, gb['mean'], marker='o', color=NAVY, linewidth=2, label='평균')
# ax.axhline(1.96, color=RED, linestyle='--', linewidth=1, label='유의 기준 (t=1.96)')
# ax.set_xticks(ks)
# ax.set_xlabel('지표에 포함한 항목 수'); ax.set_ylabel('신호 강도 (t)')
# ax.set_title('그림 17: 16,383개 항목 조합의 신호 강도 분포', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# inc = np.load('combo_inc_mask.npy')
# tv = res['t'].to_numpy()
# ok_t = ~np.isnan(tv)
# contrib = {}
# for j, c in enumerate(FVOL_COLS):
#     contrib[c] = tv[inc[:, j] & ok_t].mean() - tv[~inc[:, j] & ok_t].mean()
# contrib = pd.Series(contrib).sort_values()
# KOR = {'actq': '유동자산', 'rectq': '매출채권', 'invtq': '재고자산', 'ppentq': '유형자산',
#        'atq': '자산총계', 'lctq': '유동부채', 'dlcq': '단기차입금', 'apq': '매입채무',
#        'dlttq': '장기차입금', 'ltq': '부채총계', 'seqq': '자본총계',
#        'xoprq': '영업비용', 'cogsq': '매출원가', 'xsgaq': '판매관리비'}
# labels = [KOR[c] for c in contrib.index]
# colors10 = [RED if v < 0 else NAVY for v in contrib.values]
# fig, ax = plt.subplots(figsize=(9, 4.8))
# ax.barh(range(len(contrib)), contrib.values, color=colors10)
# ax.axvline(0, color='grey', linewidth=0.8)
# ax.set_yticks(range(len(contrib))); ax.set_yticklabels(labels)
# ax.set_xlabel('신호 기여도 (포함 시 - 제외 시 평균 t 차이)')
# ax.set_title('그림 18: 재무 항목별 신호 기여도', fontweight='bold', color=NAVY)
# ax.grid(axis='x', alpha=0.3)
# plt.tight_layout(); plt.show()



# # 시각화

# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False
# NAVY = '#1F3864'
# RED = '#C00000'

# financials = pd.read_parquet('financials.parquet')
# prices = pd.read_parquet('prices.parquet')

# fv, ind = compute_fvol(financials, deflator='saleq')
# fv, col = finalize_fvol(fv, ind, suffix='')

# sig = fv[['code', 'date', 'FVOL']].dropna().copy()
# sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1

# px = prices.copy()
# px['ret'] = px['ret'] / 100.0
# px['ym'] = pd.PeriodIndex(px['date'], freq='M')

# form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
# form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

# f = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
# f = f[f['price'] >= 1000].copy()
# f['decile'] = np.ceil(f.groupby('form_qtr')['FVOL'].rank(pct=True) * 10).clip(1, 10).astype(int)
# f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

# holds = []
# for k in [1, 2, 3]:
#     h = f[['code', 'form_ym', 'decile']].copy()
#     h['hold_ym'] = h['form_ym'] + k
#     holds.append(h)
# hold = pd.concat(holds, ignore_index=True)

# m = px[['code', 'ym', 'ret']].dropna().merge(
#     hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
# m = m.drop_duplicates(['code', 'form_ym', 'ym'])

# ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='r')
# wide = ew.pivot(index='ym', columns='decile', values='r')
# lh_m = (wide[1] - wide[10]).dropna()

# yr = lh_m.copy()
# yr.index = yr.index.year
# ann = yr.groupby(level=0).mean() * 100
# pos = yr.groupby(level=0).apply(lambda g: (g > 0).mean())
# lo_y = wide[1].dropna().copy(); lo_y.index = lo_y.index.year
# hi_y = wide[10].dropna().copy(); hi_y.index = hi_y.index.year
# lo_ann = lo_y.groupby(level=0).mean() * 100
# hi_ann = hi_y.groupby(level=0).mean() * 100
# years = ann.index.values

# fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7.8), sharex=True,
#                                gridspec_kw={'height_ratios': [1.3, 1]})
# fig.suptitle('그림 19: 연도별 성과 분해', fontweight='bold', color=NAVY, fontsize=13)

# # 상단: 연도별 격차 (승=NAVY, 패=RED) + 양수월 주석
# colors = [NAVY if v > 0 else RED for v in ann.values]
# ax1.bar(years, ann.values, color=colors, width=0.65)
# ax1.axhline(0, color='grey', linewidth=0.8)
# for x, v, p in zip(years, ann.values, pos.values):
#     ax1.annotate(f'{p:.0%}', (x, v), textcoords='offset points',
#                  xytext=(0, 4 if v > 0 else -11), ha='center', fontsize=7, color='#555555')
# ax1.set_ylabel('격차 (%/월)')
# ax1.set_title('연도별 최저-최고 분위 수익률 격차 (막대 위: 양수월 비율, 파랑=승 / 빨강=패)',
#               fontsize=10, color='#333333')
# ax1.grid(axis='y', alpha=0.3)

# # 하단: Low / High 실제 수익률
# ax2.plot(years, lo_ann.values, marker='o', color=NAVY, linewidth=1.8, label='최저 FVOL 분위 (매수)')
# ax2.plot(years, hi_ann.values, marker='s', color=RED, linewidth=1.8, label='최고 FVOL 분위 (매도)')
# ax2.axhline(0, color='grey', linewidth=0.8)
# ax2.set_ylabel('평균 월수익률 (%)')
# ax2.set_xlabel('연도')
# ax2.set_title('연도별 최저·최고 분위 수익률', fontsize=10, color='#333333')
# ax2.legend(frameon=False)
# ax2.grid(alpha=0.3)
# ax2.set_xticks(years)
# ax2.tick_params(axis='x', rotation=45)

# plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

# # 시각화

# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False
# NAVY = '#1F3864'
# RED = '#C00000'
# GRAY = '#B0B0B0'

# financials = pd.read_parquet('financials.parquet')
# prices = pd.read_parquet('prices.parquet')
# market = pd.read_parquet('market.parquet')

# fv, ind = compute_fvol(financials, deflator='saleq')
# fv, col = finalize_fvol(fv, ind, suffix='')

# px = prices.copy()
# px['ret'] = px['ret'] / 100.0
# px['ym'] = pd.PeriodIndex(px['date'], freq='M')
# form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
# form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')
# ret_m = px[['code', 'ym', 'ret']].dropna()

# def lh_of(valcol):
#     s = fv[['code', 'date', valcol]].dropna().copy()
#     s['form_qtr'] = pd.PeriodIndex(s['date'], freq='Q') + 1
#     f = s.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
#     f = f[f['price'] >= 1000].copy()
#     f['decile'] = np.ceil(f.groupby('form_qtr')[valcol].rank(pct=True) * 10).clip(1, 10).astype(int)
#     f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')
#     holds = []
#     for k in [1, 2, 3]:
#         h = f[['code', 'form_ym', 'decile']].copy()
#         h['hold_ym'] = h['form_ym'] + k
#         holds.append(h)
#     hold = pd.concat(holds, ignore_index=True)
#     m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
#     m = m.drop_duplicates(['code', 'form_ym', 'ym'])
#     ew = m.groupby(['ym', 'decile'])['ret'].mean().reset_index(name='r')
#     w = ew.pivot(index='ym', columns='decile', values='r')
#     return (w[1] - w[10]).dropna()

# lh_fvol = lh_of('FVOL')
# mk = market.dropna(subset=['kret']).copy()
# mk['kret'] = mk['kret'] / 100.0
# mk['ym'] = pd.PeriodIndex(mk['date'], freq='M')
# rows = []
# for t in lh_fvol.index:
#     sub = mk[mk['ym'].isin([t - 2, t - 1, t])]['kret']
#     if len(sub) >= 20:
#         rows.append({'ym': t, 'vol': sub.std(), 'trend': (1 + sub).prod() - 1})
# regime = pd.DataFrame(rows)
# regime['zvol'] = (regime['vol'] - regime['vol'].mean()) / regime['vol'].std()
# regime['ztr'] = (regime['trend'] - regime['trend'].mean()) / regime['trend'].std()
# vol_med = regime['vol'].median()
# regime['quad'] = ((regime['vol'] > vol_med).map({True: '고변동', False: '저변동'})
#                   + (regime['trend'] > 0).map({True: '상승', False: '하락'}))
# regime['year'] = regime['ym'].dt.year
# QUADS = ['고변동하락', '저변동상승', '고변동상승', '저변동하락']
# QCOLOR = {'고변동하락': RED, '저변동상승': NAVY, '고변동상승': '#C55A11', '저변동하락': GRAY}

# zt0 = (0 - regime['trend'].mean()) / regime['trend'].std()
# zv_med = (vol_med - regime['vol'].mean()) / regime['vol'].std()
# fig, ax = plt.subplots(figsize=(9, 5.5))
# for q in QUADS:
#     s = regime[regime['quad'] == q]
#     ax.scatter(s['zvol'], s['ztr'], c=QCOLOR[q], label=q, s=28, alpha=0.75, edgecolors='white', linewidths=0.4)
# ax.axvline(zv_med, color='grey', linewidth=0.9, linestyle='--')
# ax.axhline(zt0, color='grey', linewidth=0.9, linestyle='--')
# ax.set_xlabel('시장 변동성 (표준화)'); ax.set_ylabel('시장 추세 (표준화)')
# ax.set_title('그림 20: 시장 국면 분류 (291개월)', fontweight='bold', color=NAVY)
# ax.legend(frameon=False, loc='upper right'); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# qstat = {}
# for q in QUADS:
#     d = lh_fvol.rename('lh').reset_index().merge(regime[['ym', 'quad']], on='ym').dropna()
#     s = d[d['quad'] == q]['lh']
#     t = s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))
#     qstat[q] = (s.mean() * 100, t)
# fig, ax = plt.subplots(figsize=(9, 4.5))
# vals = [qstat[q][0] for q in QUADS]
# cols = [NAVY if abs(qstat[q][1]) >= 1.96 else GRAY for q in QUADS]
# ax.bar(range(4), vals, color=cols, width=0.6)
# for i, q in enumerate(QUADS):
#     ax.annotate(f't={qstat[q][1]:.2f}', (i, vals[i]), textcoords='offset points',
#                 xytext=(0, 5 if vals[i] >= 0 else -12), ha='center', fontsize=9)
# ax.axhline(0, color='grey', linewidth=0.8)
# ax.set_xticks(range(4)); ax.set_xticklabels(QUADS)
# ax.set_ylabel('최저-최고 분위 격차 (%/월)')
# ax.set_title('그림 21: 국면별 신호 강도 (회색: 비유의)', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# yr = regime.groupby('year').agg(n=('vol', 'size')).copy()
# lhy = lh_fvol.copy(); lhy.index = lhy.index.year
# yr['lh'] = lhy.groupby(level=0).mean() * 100
# yr['승패'] = np.where(yr['lh'] > 0, '승', '패')
# comp = regime.pivot_table(index='year', columns='quad', values='vol', aggfunc='size', fill_value=0)
# for q in QUADS:
#     if q not in comp.columns: comp[q] = 0
# comp_pct = comp[QUADS].div(comp[QUADS].sum(axis=1), axis=0) * 100
# win = yr[yr['승패'] == '승'].index; los = yr[yr['승패'] == '패'].index
# win_m = [comp_pct.loc[win, q].mean() for q in QUADS]
# los_m = [comp_pct.loc[los, q].mean() for q in QUADS]
# fig, ax = plt.subplots(figsize=(9, 4.5))
# x = np.arange(4); w = 0.38
# b1 = ax.bar(x - w/2, win_m, w, color=NAVY, label='승 연도')
# b2 = ax.bar(x + w/2, los_m, w, color=RED, label='패 연도')
# for bars in (b1, b2):
#     for rect in bars:
#         h = rect.get_height()
#         ax.annotate(f'{h:.1f}', (rect.get_x() + rect.get_width()/2, h),
#                     textcoords='offset points', xytext=(0, 3), ha='center', fontsize=8.5)
# ax.set_xticks(x); ax.set_xticklabels(QUADS)
# ax.set_ylabel('국면 구성 비율 (%)')
# ax.set_title('그림 22: 승 연도와 패 연도의 국면 구성', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# del_bad = set(delisted[delisted['reason_cat'].isin(['감사의견', '정리절차', '자본잠식'])]['code'])
# mrg_set = set(delisted[delisted['reason_cat'] == '합병·해산']['code'])
# del_q = {}
# for _, r in delisted.iterrows():
#     if pd.notna(r['del_date']):
#         pq = pd.Period(r['del_date'], freq='Q'); del_q[r['code']] = pq.year * 4 + pq.quarter - 1
# d = fv[fv['FVOL'].notna()][['code', 'date', 'FVOL']].copy()
# pq = pd.PeriodIndex(d['date'], freq='Q'); d['qidx'] = pq.year * 4 + (pq.quarter - 1)
# d['dec'] = d.groupby('date')['FVOL'].transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop')) + 1
# def within8(code, qi, tset):
#     dq = del_q.get(code)
#     return int(dq is not None and code in tset and 0 < dq - qi <= 8)
# d['bad8'] = [within8(c, q, del_bad) for c, q in zip(d['code'], d['qidx'])]
# d['mrg8'] = [within8(c, q, mrg_set) for c, q in zip(d['code'], d['qidx'])]
# bad_rate = [d[d['dec'] == dc]['bad8'].mean() * 100 for dc in range(1, 11)]
# mrg_rate = [d[d['dec'] == dc]['mrg8'].mean() * 100 for dc in range(1, 11)]
# fig, ax = plt.subplots(figsize=(9, 4.5))
# x = np.arange(1, 11); w = 0.4
# ax.bar(x - w/2, bad_rate, w, color=RED, label='부실 폐지')
# ax.bar(x + w/2, mrg_rate, w, color=GRAY, label='합병 폐지')
# ax.set_xticks(x); ax.set_xlabel('FVOL 분위 (10=최고 변동성)'); ax.set_ylabel('8분기 내 폐지율 (%)')
# ax.set_title('그림 23: FVOL 분위별 부실·합병 폐지율', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# surv_mean = d[~d['code'].isin(del_bad)]['FVOL'].mean()
# traj = []
# for off in range(-12, 1):
#     vals = []
#     for c in del_bad:
#         dq = del_q.get(c)
#         if dq is None: continue
#         sub = d[(d['code'] == c) & (d['qidx'] == dq + off)]
#         if len(sub): vals.append(sub['FVOL'].iloc[0])
#     traj.append(np.mean(vals) if vals else np.nan)
# fig, ax = plt.subplots(figsize=(9, 4.5))
# offs = list(range(-12, 1))
# ax.plot(offs, traj, marker='o', color=RED, linewidth=2, label='부실 폐지 종목')
# ax.axhline(surv_mean, color=NAVY, linewidth=2, linestyle='--', label=f'생존 기업 평균 ({surv_mean:.2f})')
# ax.set_ylim(0.45, 0.87)
# ax.set_xlabel('폐지까지 남은 분기'); ax.set_ylabel('FVOL 순위 (0~1)')
# ax.set_title('그림 24: 폐지 전 FVOL 순위 궤적', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# KOR = {'actq': '유동자산', 'rectq': '매출채권', 'invtq': '재고자산', 'ppentq': '유형자산',
#        'atq': '자산총계', 'lctq': '유동부채', 'dlcq': '단기차입금', 'apq': '매입채무',
#        'dlttq': '장기차입금', 'ltq': '부채총계', 'seqq': '자본총계',
#        'xoprq': '영업비용', 'cogsq': '매출원가', 'xsgaq': '판매관리비'}
# r25 = []
# for c in FVOL_COLS:
#     lh = lh_of(f'fvol_{c}')
#     t = lh.mean() / (lh.std(ddof=1) / np.sqrt(len(lh)))
#     r25.append((KOR[c], lh.mean() * 100, t))
# r25.sort(key=lambda z: z[1])
# labels = [z[0] for z in r25]; vals = [z[1] for z in r25]; tvals = [z[2] for z in r25]
# cols = [NAVY if abs(z[2]) >= 1.96 else GRAY for z in r25]
# fig, ax = plt.subplots(figsize=(9, 5.5))
# ax.barh(range(len(r25)), vals, color=cols)
# xmax = max(vals)
# for i, (v, t) in enumerate(zip(vals, tvals)):
#     ax.annotate(f't={t:.2f}', (v, i), textcoords='offset points', xytext=(4, 0),
#                 va='center', ha='left', fontsize=8.5,
#                 color=(NAVY if abs(t) >= 1.96 else GRAY))
# ax.set_yticks(range(len(r25))); ax.set_yticklabels(labels)
# ax.set_xlim(0, xmax * 1.15)
# ax.set_xlabel('단독 최저-최고 분위 격차 (%/월)')
# ax.set_title('그림 25: 재무 항목별 단독 신호 (진한색: 유의)', fontweight='bold', color=NAVY)
# ax.grid(axis='x', alpha=0.3)
# plt.tight_layout(); plt.show()

# # 시각화

# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False
# NAVY = '#1F3864'
# RED = '#C00000'
# GRAY = '#B0B0B0'
# PALETTE = ['#1F3864', '#D3834D', '#CF4040', '#576A8B', '#7BA05B', '#8E6C8A', '#C0A040']

# financials = pd.read_parquet('financials.parquet')
# prices = pd.read_parquet('prices.parquet')
# ivol = pd.read_parquet('ivol_quarterly.parquet')
# mktf = pd.read_parquet('mkt_factor.parquet')
# jkp = pd.read_parquet('jkp_factors.parquet')
# fvret = pd.read_parquet('fvol_factor_ret.parquet')

# fv, ind = compute_fvol(financials, deflator='saleq')
# fv, col = finalize_fvol(fv, ind, suffix='')

# sig = fv[['code', 'date', 'FVOL']].dropna().copy()
# sig['form_qtr'] = pd.PeriodIndex(sig['date'], freq='Q') + 1
# sig['prev_qtr'] = sig['form_qtr'] - 1
# sig = sig.merge(ivol.rename(columns={'qtr': 'prev_qtr'}), on=['code', 'prev_qtr'], how='left')

# px = prices.copy()
# px['ret'] = px['ret'] / 100.0
# px['ym'] = pd.PeriodIndex(px['date'], freq='M')
# form_px = px[px['ym'].dt.month.isin([3, 6, 9, 12])][['code', 'ym', 'price']].copy()
# form_px['form_qtr'] = form_px['ym'].dt.asfreq('Q')

# f0 = sig.merge(form_px[['code', 'form_qtr', 'price']], on=['code', 'form_qtr'], how='inner')
# f0 = f0[f0['price'] >= 1000].copy()
# f = f0.dropna(subset=['ivol']).copy()
# f['fq'] = f.groupby('form_qtr')['FVOL'].transform(lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
# f['iq'] = f.groupby('form_qtr')['ivol'].transform(lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')) + 1
# f = f.dropna(subset=['fq', 'iq'])
# f['fq'] = f['fq'].astype(int); f['iq'] = f['iq'].astype(int)
# f['form_ym'] = f['form_qtr'].dt.asfreq('M', how='end')

# holds = []
# for k in range(1, 25):
#     h = f[['code', 'form_ym', 'fq', 'iq']].copy()
#     h['hold_ym'] = h['form_ym'] + k
#     h['qt'] = (k - 1) // 3 + 1
#     holds.append(h)
# hold = pd.concat(holds, ignore_index=True)
# ret_m = px[['code', 'ym', 'ret']].dropna()
# m = ret_m.merge(hold, left_on=['code', 'ym'], right_on=['code', 'hold_ym'], how='inner')
# m = m.drop_duplicates(['code', 'form_ym', 'ym', 'qt'])
# cell = m.groupby(['qt', 'ym', 'iq', 'fq'])['ret'].mean().reset_index(name='r')

# def tstat(s):
#     s = s.dropna()
#     return s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))

# grid = (cell[cell['qt'] == 1].groupby(['iq', 'fq'])['r'].mean() * 100).unstack('fq')
# fig, ax = plt.subplots(figsize=(7.5, 6))
# im = ax.imshow(grid.values, cmap='RdYlBu_r', aspect='auto')
# ax.set_xticks(range(5)); ax.set_xticklabels([f'{i}' for i in range(1, 6)])
# ax.set_yticks(range(5)); ax.set_yticklabels([f'{i}' for i in range(1, 6)])
# ax.set_xlabel('FVOL 분위 (1=최저 → 5=최고)'); ax.set_ylabel('IVOL 분위 (1=최저 → 5=최고)')
# for i in range(5):
#     for j in range(5):
#         ax.text(j, i, f'{grid.values[i, j]:.2f}', ha='center', va='center', fontsize=9,
#                 color='white' if abs(grid.values[i, j]) > 1.5 else 'black')
# ax.set_title('그림 26: FVOL×IVOL 5×5 이중정렬 구간별 평균 수익률 (Q1, %/월)', fontweight='bold', color=NAVY)
# fig.colorbar(im, ax=ax, shrink=0.8, label='월 평균 수익률 (%)')
# plt.tight_layout(); plt.show()

# # ===== 그림 27: 한 지표 통제 후 다른 지표 Low-High, 보유 분기별 =====
# rowsA, rowsB = [], []
# for qt in range(1, 9):
#     sub = cell[cell['qt'] == qt]
#     la = []
#     for v in range(1, 6):
#         w = sub[sub['iq'] == v].pivot(index='ym', columns='fq', values='r')
#         if 1 in w.columns and 5 in w.columns:
#             la.append((w[1] - w[5]).rename(v))
#     lb = []
#     for v in range(1, 6):
#         w = sub[sub['fq'] == v].pivot(index='ym', columns='iq', values='r')
#         if 1 in w.columns and 5 in w.columns:
#             lb.append((w[1] - w[5]).rename(v))
#     a = pd.concat(la, axis=1).mean(axis=1).dropna()
#     b = pd.concat(lb, axis=1).mean(axis=1).dropna()
#     rowsA.append((a.mean() * 100, tstat(a)))
#     rowsB.append((b.mean() * 100, tstat(b)))
# qs = list(range(1, 9))
# aM = [x[0] for x in rowsA]; aT = [x[1] for x in rowsA]
# bM = [x[0] for x in rowsB]; bT = [x[1] for x in rowsB]
# fig, ax = plt.subplots(figsize=(9, 4.6))
# w = 0.38
# ax.bar([q - w/2 for q in qs], aM, w, color=NAVY, label='IVOL 통제 후 FVOL')
# ax.bar([q + w/2 for q in qs], bM, w, color=GRAY, label='FVOL 통제 후 IVOL')
# for q, mv, tv in zip(qs, aM, aT):
#     ax.text(q - w/2, mv, f'{tv:.1f}', ha='center', va='bottom' if mv >= 0 else 'top', fontsize=7)
# for q, mv, tv in zip(qs, bM, bT):
#     ax.text(q + w/2, mv, f'{tv:.1f}', ha='center', va='bottom' if mv >= 0 else 'top', fontsize=7)
# ax.axhline(0, color='black', linewidth=0.6)
# ax.set_xticks(qs); ax.set_xticklabels([f'Q{q}' for q in qs])
# ax.set_xlabel('보유 분기'); ax.set_ylabel('최저-최고 격차 (%/월)')
# ax.set_title('그림 27: 한 지표를 통제한 뒤 다른 지표의 격차', fontweight='bold', color=NAVY)
# ax.legend(frameon=False); ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# dfp = fvret[['ym', 'fvol_lh']].merge(mktf[['ym', 'mktrf_ew']], on='ym', how='inner').merge(jkp, on='ym', how='inner')
# def ols(y, X):
#     X1 = np.column_stack([np.ones(len(X)), X])
#     b, _, _, _ = np.linalg.lstsq(X1, y, rcond=None)
#     r = y - X1 @ b; n, k = X1.shape
#     se = np.sqrt(np.diag((r @ r) / (n - k) * np.linalg.inv(X1.T @ X1)))
#     return b, b / se

# facs7 = ['market_equity', 'be_me', 'ope_be', 'at_gr1', 'ni_be', 'niq_be', 'size_thm']
# names7 = ['규모(시총)', '가치(be_me)', '수익성(ope_be)', '투자(at_gr1)', 'ROE(ni_be)', 'ROE(niq_be)', '규모(종합)']
# jj = jkp.dropna(subset=['be_me']).copy().sort_values('ym')
# fig, ax = plt.subplots(figsize=(9.2, 4.8))
# for fac, nm, col_ in zip(facs7, names7, PALETTE):
#     s = jj[['ym', fac]].dropna()
#     cum = (1 + s[fac]).cumprod()
#     ax.plot(s['ym'].dt.to_timestamp(), cum.values, label=nm, color=col_, linewidth=1.5)
# ax.set_yscale('log')
# ax.set_xlabel('연도'); ax.set_ylabel('누적수익률 (로그, 시작=1)')
# ax.set_title('그림 28: 검증에 사용된 JKP 한국 팩터의 누적수익률', fontweight='bold', color=NAVY)
# ax.legend(frameon=False, ncol=2, fontsize=8); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

# MODELS = {
#     'FF5\n(기본)': ['mktrf_ew', 'market_equity', 'be_me', 'ope_be', 'at_gr1'],
#     'FF5\n(강건성)': ['mktrf_ew', 'size_thm', 'be_me', 'ope_be', 'at_gr1'],
#     'q\n(기본)': ['mktrf_ew', 'market_equity', 'at_gr1', 'ni_be'],
#     'q\n(강건성)': ['mktrf_ew', 'market_equity', 'at_gr1', 'niq_be'],
# }
# anames, avals, atvals = [], [], []
# for nm, facs in MODELS.items():
#     sub = dfp[['fvol_lh'] + facs].dropna()
#     b, t = ols(sub['fvol_lh'].values, sub[facs].values)
#     anames.append(nm); avals.append(b[0] * 100); atvals.append(t[0])
# fig, ax = plt.subplots(figsize=(8, 4.6))
# colors = [NAVY if abs(t) >= 1.96 else GRAY for t in atvals]
# bars = ax.bar(range(len(anames)), avals, color=colors, width=0.6)
# for i, (v, t) in enumerate(zip(avals, atvals)):
#     ax.text(i, v, f't={t:.2f}', ha='center', va='bottom', fontsize=9)
# ax.axhline(0, color='black', linewidth=0.6)
# ax.set_xticks(range(len(anames))); ax.set_xticklabels(anames)
# ax.set_ylabel('스패닝 α (%/월)')
# ax.set_title('그림 29: 모형별 스패닝 절편 α', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# ff5 = ['mktrf_ew', 'market_equity', 'be_me', 'ope_be', 'at_gr1']
# ff5nm = ['시장', '규모', '가치', '수익성', '투자']
# sub = dfp[['fvol_lh'] + ff5].dropna()
# b, t = ols(sub['fvol_lh'].values, sub[ff5].values)
# betas = b[1:]; tvs = t[1:]
# fig, ax = plt.subplots(figsize=(8, 4.6))
# colors = [NAVY if abs(tv) >= 1.96 else GRAY for tv in tvs]
# ax.bar(range(len(ff5nm)), betas, color=colors, width=0.6)
# for i, (bv, tv) in enumerate(zip(betas, tvs)):
#     ax.text(i, bv, f't={tv:.1f}', ha='center', va='bottom' if bv >= 0 else 'top', fontsize=9)
# ax.axhline(0, color='black', linewidth=0.6)
# ax.set_xticks(range(len(ff5nm))); ax.set_xticklabels(ff5nm)
# ax.set_ylabel('팩터 노출 (회귀계수)')
# ax.set_title('그림 30: FVOL 격차의 팩터 노출', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()

# ADDVAL = {
#     'q\n(기본)': ['mktrf_ew', 'market_equity', 'at_gr1', 'ni_be'],
#     'q + 가치': ['mktrf_ew', 'market_equity', 'at_gr1', 'ni_be', 'be_me'],
#     'FF5\n(기본)': ['mktrf_ew', 'market_equity', 'be_me', 'ope_be', 'at_gr1'],
# }
# vnames, vvals, vtvals = [], [], []
# for nm, facs in ADDVAL.items():
#     sub = dfp[['fvol_lh'] + facs].dropna()
#     b, t = ols(sub['fvol_lh'].values, sub[facs].values)
#     vnames.append(nm); vvals.append(b[0] * 100); vtvals.append(t[0])
# fig, ax = plt.subplots(figsize=(7.5, 4.6))
# colors = [NAVY if abs(t) >= 1.96 else GRAY for t in vtvals]
# ax.bar(range(len(vnames)), vvals, color=colors, width=0.55)
# for i, (v, t) in enumerate(zip(vvals, vtvals)):
#     ax.text(i, v, f't={t:.2f}', ha='center', va='bottom', fontsize=9)
# ax.axhline(0, color='black', linewidth=0.6)
# ax.set_xticks(range(len(vnames))); ax.set_xticklabels(vnames)
# ax.set_ylabel('스패닝 α (%/월)')
# ax.set_title('그림 31: 가치 팩터 추가에 따른 α 변화', fontweight='bold', color=NAVY)
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout(); plt.show()